# Final Grammar–KT dataset results

This read-only notebook takes a completed Grammar–KT data-folder path and displays the retained result of every pipeline stage as pandas tables. Edit `DATA_FOLDER` below or set the `GRAMMAR_KT_DATA_FOLDER` environment variable before execution. Relative paths are resolved from the repository root.

Large learner-event and KT-prediction artifacts are summarized and sampled rather than printed in full. Raw source, canonical, item, fold, KC, and metric tables are shown completely at the default row limit. Private simulator truth and superseded pre-curation artifacts are deliberately excluded.

In [1]:
import os

DATA_FOLDER = os.environ.get('GRAMMAR_KT_DATA_FOLDER', 'data/grammar_kt_medium_v1')
TABLE_ROW_LIMIT = 200
EVENT_SAMPLE_ROWS = 20
PREDICTION_SAMPLE_ROWS_PER_TECHNIQUE = 8

## 0. Load and verify the completed dataset

The artifact inventory is the notebook's input contract. Missing final-stage artifacts fail immediately instead of silently producing a partial result.

In [2]:
import json
from pathlib import Path

import pandas as pd
import yaml
from IPython.display import Markdown, display

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 180)

cwd = Path.cwd().resolve()
ROOT = next(
    (candidate for candidate in (cwd, *cwd.parents) if (candidate / 'modules').is_dir()),
    cwd,
)
folder = Path(DATA_FOLDER).expanduser()
DATA_DIR = (folder if folder.is_absolute() else ROOT / folder).resolve()

def read_json(relative_path):
    return json.loads((DATA_DIR / relative_path).read_text(encoding='utf-8'))

def read_jsonl(relative_path, *, nrows=None):
    return pd.read_json(DATA_DIR / relative_path, lines=True, compression='infer', nrows=nrows)

def read_yaml(relative_path):
    with (DATA_DIR / relative_path).open(encoding='utf-8') as handle:
        return yaml.safe_load(handle)

def compact(value):
    if isinstance(value, (dict, list, tuple)):
        return json.dumps(value, ensure_ascii=False, sort_keys=True)
    return value

STAGE_TABLES = {}

def show_table(title, frame, *, limit=TABLE_ROW_LIMIT):
    frame = frame.reset_index(drop=True).copy()
    STAGE_TABLES[title] = frame
    display(Markdown(f'**{title}** — {len(frame):,} row(s)'))
    display(frame.head(limit))
    return frame

artifact_spec = [
    ('metadata', 'manifest.json'),
    ('metadata', 'finalization_manifest.json'),
    ('source', 'source/descriptors.jsonl'),
    ('normalisation', 'normalisation/mappings.jsonl'),
    ('canonicalisation', 'canonical/cells.jsonl'),
    ('canonicalisation', 'canonical/source_cell_relations.jsonl'),
    ('item generation', 'items/generation_attempts.jsonl'),
    ('item generation', 'items/candidates.jsonl'),
    ('item validation', 'items/curated_validation.jsonl'),
    ('fixed item bank', 'items/selected_bank.jsonl'),
    ('grammar fold', 'fold/assignments.jsonl'),
    ('KC candidates', 'kc/candidate_inventory.json'),
    ('learner evidence', 'simulation/events.jsonl.gz'),
    ('KC selection', 'kc/selection_trace.json'),
    ('KC stability', 'kc/selection_stability.json'),
    ('projection', 'kc/projections/automated.jsonl'),
    ('Q-matrix', 'kc/q_matrices/automated.csv'),
    ('KT', 'kt/automated/predictions.jsonl.gz'),
    ('evaluation', 'evaluation/automated/results.json'),
    ('uncertainty', 'evaluation/paired_logistic.json'),
]
artifact_inventory = pd.DataFrame([
    {
        'stage': stage,
        'relative_path': relative_path,
        'exists': (DATA_DIR / relative_path).is_file(),
        'size_bytes': (DATA_DIR / relative_path).stat().st_size if (DATA_DIR / relative_path).is_file() else pd.NA,
    }
    for stage, relative_path in artifact_spec
])
missing = artifact_inventory.loc[~artifact_inventory['exists'], 'relative_path'].tolist()
if missing:
    raise FileNotFoundError(f'Incomplete Grammar–KT data folder {DATA_DIR}: missing {missing}')

pipeline_index = pd.DataFrame([
    {'stage': 1, 'name': 'Typed grammar source', 'principal_artifact': 'source/descriptors.jsonl', 'scientific boundary': 'resource-specific evidence'},
    {'stage': 2, 'name': 'Normalisation', 'principal_artifact': 'normalisation/mappings.jsonl', 'scientific boundary': 'constrained source → canonical proposals'},
    {'stage': 3, 'name': 'Canonicalisation', 'principal_artifact': 'canonical/cells.jsonl', 'scientific boundary': 'deduplicated GrammarCells'},
    {'stage': 4, 'name': 'Item generation', 'principal_artifact': 'items/candidates.jsonl', 'scientific boundary': 'no fold, outcome, KC, or KT input'},
    {'stage': 5, 'name': 'Validation and fixed bank', 'principal_artifact': 'items/selected_bank.jsonl', 'scientific boundary': 'independent judgments before learner evidence'},
    {'stage': 6, 'name': 'Semantic grammar fold', 'principal_artifact': 'fold/assignments.jsonl', 'scientific boundary': 'outcome-free grammar regimes'},
    {'stage': 7, 'name': 'KC candidate generation', 'principal_artifact': 'kc/candidate_inventory.json', 'scientific boundary': 'development grammar and fixed items only'},
    {'stage': 8, 'name': 'Learner evidence', 'principal_artifact': 'simulation/events.jsonl.gz', 'scientific boundary': 'development acquisition, then frozen probes'},
    {'stage': 9, 'name': 'Automated KC selection', 'principal_artifact': 'kc/selection_trace.json', 'scientific boundary': 'development train/validation outcomes only'},
    {'stage': 10, 'name': 'Frozen projection / Q-matrix', 'principal_artifact': 'kc/q_matrices/automated.csv', 'scientific boundary': 'selected policy frozen before holdout projection'},
    {'stage': 11, 'name': 'Knowledge tracing', 'principal_artifact': 'kt/*/predictions.jsonl.gz', 'scientific boundary': 'identical event stream across representations'},
    {'stage': 12, 'name': 'Evaluation and uncertainty', 'principal_artifact': 'evaluation/*/results.json', 'scientific boundary': 'held-out probes and learner-paired intervals'},
])

show_table('Resolved input', pd.DataFrame([{'DATA_FOLDER': str(DATA_DIR), 'exists': DATA_DIR.is_dir()}]))
show_table('Required artifact inventory', artifact_inventory)
show_table('Pipeline stage index', pipeline_index)

**Resolved input** — 1 row(s)

,DATA_FOLDER,exists
0,/home/abdullah/grammar_kt_harness/data/grammar_kt_medium_v1,True


**Required artifact inventory** — 20 row(s)

,stage,relative_path,exists,size_bytes
0,metadata,manifest.json,True,7998
1,metadata,finalization_manifest.json,True,2663
2,source,source/descriptors.jsonl,True,76932
3,normalisation,normalisation/mappings.jsonl,True,37015
4,canonicalisation,canonical/cells.jsonl,True,6029
5,canonicalisation,canonical/source_cell_relations.jsonl,True,6297
6,item generation,items/generation_attempts.jsonl,True,29521
7,item generation,items/candidates.jsonl,True,39630
8,item validation,items/curated_validation.jsonl,True,118099
9,fixed item bank,items/selected_bank.jsonl,True,40763


**Pipeline stage index** — 12 row(s)

,stage,name,principal_artifact,scientific boundary
0,1,Typed grammar source,source/descriptors.jsonl,resource-specific evidence
1,2,Normalisation,normalisation/mappings.jsonl,constrained source → canonical proposals
2,3,Canonicalisation,canonical/cells.jsonl,deduplicated GrammarCells
3,4,Item generation,items/candidates.jsonl,"no fold, outcome, KC, or KT input"
4,5,Validation and fixed bank,items/selected_bank.jsonl,independent judgments before learner evidence
5,6,Semantic grammar fold,fold/assignments.jsonl,outcome-free grammar regimes
6,7,KC candidate generation,kc/candidate_inventory.json,development grammar and fixed items only
7,8,Learner evidence,simulation/events.jsonl.gz,"development acquisition, then frozen probes"
8,9,Automated KC selection,kc/selection_trace.json,development train/validation outcomes only
9,10,Frozen projection / Q-matrix,kc/q_matrices/automated.csv,selected policy frozen before holdout projection


,stage,name,principal_artifact,scientific boundary
0,1,Typed grammar source,source/descriptors.jsonl,resource-specific evidence
1,2,Normalisation,normalisation/mappings.jsonl,constrained source → canonical proposals
2,3,Canonicalisation,canonical/cells.jsonl,deduplicated GrammarCells
3,4,Item generation,items/candidates.jsonl,"no fold, outcome, KC, or KT input"
4,5,Validation and fixed bank,items/selected_bank.jsonl,independent judgments before learner evidence
5,6,Semantic grammar fold,fold/assignments.jsonl,outcome-free grammar regimes
6,7,KC candidate generation,kc/candidate_inventory.json,development grammar and fixed items only
7,8,Learner evidence,simulation/events.jsonl.gz,"development acquisition, then frozen probes"
8,9,Automated KC selection,kc/selection_trace.json,development train/validation outcomes only
9,10,Frozen projection / Q-matrix,kc/q_matrices/automated.csv,selected policy frozen before holdout projection


In [3]:
manifest = read_json('manifest.json')
finalization = read_json('finalization_manifest.json')

dataset_overview = pd.DataFrame([{
    'dataset_id': finalization['dataset_id'],
    'item_bank_status': manifest['status'],
    'downstream_status': finalization['status'],
    'source_descriptors': manifest['static_summary']['descriptors'],
    'canonical_cells': finalization['scale']['cells'],
    'selected_items': finalization['scale']['selected_items'],
    'learners': finalization['scale']['learners'],
    'events': finalization['scale']['events'],
    'simulation_seed': finalization['simulation']['seed'],
    'latent_world': finalization['simulation']['world_id'],
    'KC_candidate_design': finalization['kc']['candidate_design_id'],
    'KC_selection_method': finalization['kc']['selection_id'],
    'KT_protocol': finalization['kt_protocol_id'],
    'evaluation_protocol': finalization['evaluation_protocol_id'],
}])
model_table = pd.DataFrame([
    {'role': role, 'model_or_source': model}
    for role, model in manifest['models'].items()
])
show_table('Final dataset overview', dataset_overview)
show_table('Model settings retained in the dataset', model_table)

**Final dataset overview** — 1 row(s)

,dataset_id,item_bank_status,downstream_status,source_descriptors,canonical_cells,selected_items,learners,events,simulation_seed,latent_world,KC_candidate_design,KC_selection_method,KT_protocol,evaluation_protocol
0,grammar_kt_medium_v1,fixed_item_bank_complete,downstream_finalized,139,24,44,1000,204000,20260827,phase4_mixed_v1,schema_structural_candidates_v1,forward_predictive_parsimony_v1,simple_online_kt_v1,grammar_kt_evaluation_v1


**Model settings retained in the dataset** — 4 row(s)

,role,model_or_source
0,normalisation,"retained gpt-5.6-sol, medium, 2026-08-20"
1,generation,gpt-5.6-sol
2,validation,gpt-5.6-terra
3,reasoning_effort,medium


,role,model_or_source
0,normalisation,"retained gpt-5.6-sol, medium, 2026-08-20"
1,generation,gpt-5.6-sol
2,validation,gpt-5.6-terra
3,reasoning_effort,medium


## 1. Typed grammar source

The source table contains the retained English Grammar Profile descriptors. Examples remain evidence attached to source records; they are not learner items.

In [4]:
source_rows = read_jsonl('source/descriptors.jsonl')
source_display = source_rows.copy()
source_display['example_count'] = source_display['examples'].map(len)
source_display['examples'] = source_display['examples'].map(compact)
source_group_counts = pd.concat([
    source_rows.groupby('cefr', dropna=False).size().rename('descriptors').reset_index().rename(columns={'cefr': 'value'}).assign(grouping='CEFR'),
    source_rows.groupby('supercategory', dropna=False).size().rename('descriptors').reset_index().rename(columns={'supercategory': 'value'}).assign(grouping='supercategory'),
    source_rows.groupby('subcategory', dropna=False).size().rename('descriptors').reset_index().rename(columns={'subcategory': 'value'}).assign(grouping='subcategory'),
], ignore_index=True)[['grouping', 'value', 'descriptors']]
source_summary = pd.DataFrame([{
    'descriptors': len(source_rows),
    'unique_source_ids': source_rows['source_id'].nunique(),
    'CEFR_levels': source_rows['cefr'].nunique(),
    'supercategories': source_rows['supercategory'].nunique(),
    'subcategories': source_rows['subcategory'].nunique(),
    'source_examples': source_rows['examples'].map(len).sum(),
}])
show_table('Source summary', source_summary)
show_table('Source coverage by declared grouping', source_group_counts.sort_values(['grouping', 'descriptors', 'value'], ascending=[True, False, True]))
show_table('Typed source descriptors', source_display)

**Source summary** — 1 row(s)

,descriptors,unique_source_ids,CEFR_levels,supercategories,subcategories,source_examples
0,139,139,6,8,29,392


**Source coverage by declared grouping** — 43 row(s)

,grouping,value,descriptors
0,CEFR,A2,63
1,CEFR,B1,38
2,CEFR,A1,18
3,CEFR,B2,16
4,CEFR,C2,3
5,CEFR,C1,1
6,subcategory,passives: form,16
7,subcategory,imperatives,10
8,subcategory,interrogatives,10
9,subcategory,present simple,9


**Typed source descriptors** — 139 row(s)

,source_id,supercategory,subcategory,guideword,can_do,examples,cefr,example_count
0,1741163708336x930002080700909000,CLAUSES,interrogatives,FORM: AFFIRMATIVE INTERROGATIVE,Can form interrogative clauses ('yes/no' forms) of main lexical verbs with auxiliary 'do'.,"[""Do you remember Julie? (Brazil; A2 WAYSTAGE; 2004; Portuguese; Pass)"", ""Did you buy a new mobile phone? (Spain; A2...",A2,2
1,1741163708343x587422418001030500,CLAUSES,interrogatives,"FORM: 'WH-' INTERROGATIVE, SUBJECT","Can form questions with a 'wh-' word as subject, without an auxiliary verb.","[""What happened? (Spanish - Latin American, B1 THRESHOLD)"", ""Who cares? (Spanish - Latin American, B1 THRESHOLD)"", ""...",B1,3
2,1741163715033x199281404596736400,QUESTIONS,yes/no,FORM: LEXICAL VERBS WITH 'DO',Can use auxiliary 'do' + subject + main verb to form 'yes/no' questions.,"[""My favourite song is 'Viva la vida'. Do you know it? (Spain; A2 WAYSTAGE; 2009; Spanish - European; Pass)"", ""My be...",A2,4
3,1741163715031x898125206259626100,QUESTIONS,wh-,FORM: MAIN VERB 'BE',Can use 'wh-'words + main verb 'be' + subject to form 'wh-' questions.,"[""How was your dinner yesterday? (Mexico; A2 WAYSTAGE; 2009; Spanish - Latin American; Pass)"", ""How are you, my frie...",A2,3
4,1741163708330x678606711578176000,CLAUSES,declarative,"FORM/USE: AUXILIARY 'DO', FOR EMPHASIS","Can use the auxiliary verb 'do' in an affirmative declarative clause, for emphasis and affirmation.","[""Yes, I do have a favorite restaurant. (Mexico; B1 THRESHOLD; 2005; Spanish - Latin American; Pass)"", ""I do miss yo...",B1,4
...,...,...,...,...,...,...,...,...
134,1741163710395x413989918547746700,MODALITY,dare,FORM: NEGATIVE,Can use negative form dare not and daren't + infinitive without to.,"[""[talking about a restaurant] But many local people dare not go to Lily, because it is too expensive. (Taiwan; B2 V...",B2,1
135,1741163710395x580382123224487700,MODALITY,dare,FORM: AFFIRMATIVE,Can use affirmative form dare + infinitive without to.,"[""I can cook if you dare eat it! (Taiwan; B2 VANTAGE; 2001; Chinese; Pass)""]",B2,1
136,1741163712056x895221609473611600,PASSIVES,get and have,FORM: 'GET' + '-ED',Can form the 'get'-passive with a range of forms of 'get' + past participles.,"[""[talking about a mirror] I bought it because mine got broken, so I need it to see myself in it. (Mexico; B1 THRESH...",B1,5
137,1741163715620x161645047931569860,CLAUSES,conditional,"FORM/USE: PRESENT SIMPLE 'IF' CLAUSE + MODAL, FUTURE, POSSIBLE OUTCOME","Can use 'if' + present simple to introduce a possible future condition, with modal verbs in the main clause, to talk...","[""I think that it would be very good if you start going to the gym or you start cycling. (Colombia; B1 THRESHOLD; 20...",B1,3


,source_id,supercategory,subcategory,guideword,can_do,examples,cefr,example_count
0,1741163708336x930002080700909000,CLAUSES,interrogatives,FORM: AFFIRMATIVE INTERROGATIVE,Can form interrogative clauses ('yes/no' forms) of main lexical verbs with auxiliary 'do'.,"[""Do you remember Julie? (Brazil; A2 WAYSTAGE; 2004; Portuguese; Pass)"", ""Did you buy a new mobile phone? (Spain; A2...",A2,2
1,1741163708343x587422418001030500,CLAUSES,interrogatives,"FORM: 'WH-' INTERROGATIVE, SUBJECT","Can form questions with a 'wh-' word as subject, without an auxiliary verb.","[""What happened? (Spanish - Latin American, B1 THRESHOLD)"", ""Who cares? (Spanish - Latin American, B1 THRESHOLD)"", ""...",B1,3
2,1741163715033x199281404596736400,QUESTIONS,yes/no,FORM: LEXICAL VERBS WITH 'DO',Can use auxiliary 'do' + subject + main verb to form 'yes/no' questions.,"[""My favourite song is 'Viva la vida'. Do you know it? (Spain; A2 WAYSTAGE; 2009; Spanish - European; Pass)"", ""My be...",A2,4
3,1741163715031x898125206259626100,QUESTIONS,wh-,FORM: MAIN VERB 'BE',Can use 'wh-'words + main verb 'be' + subject to form 'wh-' questions.,"[""How was your dinner yesterday? (Mexico; A2 WAYSTAGE; 2009; Spanish - Latin American; Pass)"", ""How are you, my frie...",A2,3
4,1741163708330x678606711578176000,CLAUSES,declarative,"FORM/USE: AUXILIARY 'DO', FOR EMPHASIS","Can use the auxiliary verb 'do' in an affirmative declarative clause, for emphasis and affirmation.","[""Yes, I do have a favorite restaurant. (Mexico; B1 THRESHOLD; 2005; Spanish - Latin American; Pass)"", ""I do miss yo...",B1,4
...,...,...,...,...,...,...,...,...
134,1741163710395x413989918547746700,MODALITY,dare,FORM: NEGATIVE,Can use negative form dare not and daren't + infinitive without to.,"[""[talking about a restaurant] But many local people dare not go to Lily, because it is too expensive. (Taiwan; B2 V...",B2,1
135,1741163710395x580382123224487700,MODALITY,dare,FORM: AFFIRMATIVE,Can use affirmative form dare + infinitive without to.,"[""I can cook if you dare eat it! (Taiwan; B2 VANTAGE; 2001; Chinese; Pass)""]",B2,1
136,1741163712056x895221609473611600,PASSIVES,get and have,FORM: 'GET' + '-ED',Can form the 'get'-passive with a range of forms of 'get' + past participles.,"[""[talking about a mirror] I bought it because mine got broken, so I need it to see myself in it. (Mexico; B1 THRESH...",B1,5
137,1741163715620x161645047931569860,CLAUSES,conditional,"FORM/USE: PRESENT SIMPLE 'IF' CLAUSE + MODAL, FUTURE, POSSIBLE OUTCOME","Can use 'if' + present simple to introduce a possible future condition, with modal verbs in the main clause, to talk...","[""I think that it would be very good if you start going to the gym or you start cycling. (Colombia; B1 THRESHOLD; 20...",B1,3


## 2. Resource-specific normalisation

Only `complete` mappings contribute to the final canonical inventory. Partial rows may contain provisional cells, so the tables keep provisional output distinct from accepted source→cell evidence.

In [5]:
mapping_rows = read_jsonl('normalisation/mappings.jsonl')
mapping_table = mapping_rows[['source_id', 'result', 'cells', 'phase2_eligible', 'note']].copy()
mapping_table['provisional_cell_count'] = mapping_table['cells'].map(len)
mapping_table['accepted_for_canonicalisation'] = mapping_table['result'].eq('complete')
mapping_table['phase2_eligible'] = mapping_table['phase2_eligible'].map(compact)
mapping_table['cells'] = mapping_table['cells'].map(compact)
normalisation_funnel = mapping_rows.groupby('result', dropna=False).agg(
    descriptors=('source_id', 'size'),
    provisional_cells=('cells', lambda values: sum(len(value) for value in values)),
).reset_index()
normalisation_funnel['descriptor_share'] = normalisation_funnel['descriptors'] / len(mapping_rows)

complete_exploded = mapping_rows.loc[mapping_rows['result'].eq('complete'), ['source_id', 'cells']].explode('cells').dropna(subset=['cells'])
accepted_normalised_cells = pd.concat([
    complete_exploded[['source_id']].reset_index(drop=True),
    pd.json_normalize(complete_exploded['cells']).reset_index(drop=True),
], axis=1)
phase2_rows = mapping_rows[['source_id', 'phase2_eligible']].explode('phase2_eligible').dropna(subset=['phase2_eligible'])
phase2_summary = phase2_rows.groupby('phase2_eligible').agg(descriptors=('source_id', 'nunique')).reset_index()
show_table('Normalisation result funnel', normalisation_funnel.sort_values('descriptors', ascending=False))
show_table('Phase-2 eligibility by unresolved dimension', phase2_summary)
show_table('All normalisation mappings', mapping_table)
show_table('Accepted complete source→feature tuples', accepted_normalised_cells)

**Normalisation result funnel** — 4 row(s)

,result,descriptors,provisional_cells,descriptor_share
0,partial,77,86,0.553957
1,complete,44,48,0.316547
2,out_of_scope,16,0,0.115108
3,unresolved,2,0,0.014388


**Phase-2 eligibility by unresolved dimension** — 1 row(s)

,phase2_eligible,descriptors
0,tense,9


**All normalisation mappings** — 139 row(s)

,source_id,result,cells,phase2_eligible,note,provisional_cell_count,accepted_for_canonicalisation
0,1741163708336x930002080700909000,complete,"[{""aspect"": ""none"", ""clause"": ""polar_question"", ""modal"": ""none"", ""polarity"": ""positive"", ""tense"": ""present"", ""voice""...","[""tense""]",phase2 eligible: tense,2,True
1,1741163708343x587422418001030500,partial,"[{""aspect"": ""none"", ""clause"": ""subject_wh_question"", ""modal"": ""none"", ""polarity"": null, ""tense"": ""present"", ""voice"":...","[""tense""]",phase2 eligible: tense,2,False
2,1741163715033x199281404596736400,partial,"[{""aspect"": ""none"", ""clause"": ""polar_question"", ""modal"": ""none"", ""polarity"": null, ""tense"": ""present"", ""voice"": ""act...","[""tense""]",phase2 eligible: tense,2,False
3,1741163715031x898125206259626100,partial,"[{""aspect"": ""none"", ""clause"": [""subject_wh_question"", ""non_subject_wh_question""], ""modal"": ""none"", ""polarity"": null,...","[""tense""]",phase2 eligible: tense,2,False
4,1741163708330x678606711578176000,complete,"[{""aspect"": ""none"", ""clause"": ""declarative"", ""modal"": ""none"", ""polarity"": ""positive"", ""tense"": ""present"", ""voice"": ""...","[""tense""]",phase2 eligible: tense,2,True
...,...,...,...,...,...,...,...
134,1741163710395x413989918547746700,out_of_scope,[],[],Semi-modal dare is outside the central-modal ontology.,0,False
135,1741163710395x580382123224487700,out_of_scope,[],[],Dare as a semi-modal is outside the frozen schema.,0,False
136,1741163712056x895221609473611600,out_of_scope,[],[],GET-passives are explicitly out of scope.,0,False
137,1741163715620x161645047931569860,out_of_scope,[],[],Linked conditional construction is out of scope.,0,False


**Accepted complete source→feature tuples** — 48 row(s)

,source_id,tense,aspect,voice,polarity,clause,modal
0,1741163708336x930002080700909000,present,none,active,positive,polar_question,none
1,1741163708336x930002080700909000,past,none,active,positive,polar_question,none
2,1741163708330x678606711578176000,present,none,active,positive,declarative,none
3,1741163708330x678606711578176000,past,none,active,positive,declarative,none
4,1741163708330x197590942940390140,NA,none,active,positive,imperative,none
5,1741163716068x445742738976088800,NA,none,active,negative,imperative,none
6,1741163708332x482855077867024000,NA,none,active,positive,imperative,none
7,1741163708331x843997098925344500,NA,none,active,positive,imperative,none
8,1741163708332x403656410301087360,NA,none,active,negative,imperative,none
9,1741163708335x133935596064802990,NA,none,active,positive,imperative,none


,source_id,tense,aspect,voice,polarity,clause,modal
0,1741163708336x930002080700909000,present,none,active,positive,polar_question,none
1,1741163708336x930002080700909000,past,none,active,positive,polar_question,none
2,1741163708330x678606711578176000,present,none,active,positive,declarative,none
3,1741163708330x678606711578176000,past,none,active,positive,declarative,none
4,1741163708330x197590942940390140,NA,none,active,positive,imperative,none
5,1741163716068x445742738976088800,NA,none,active,negative,imperative,none
6,1741163708332x482855077867024000,NA,none,active,positive,imperative,none
7,1741163708331x843997098925344500,NA,none,active,positive,imperative,none
8,1741163708332x403656410301087360,NA,none,active,negative,imperative,none
9,1741163708335x133935596064802990,NA,none,active,positive,imperative,none


## 3. Canonical GrammarCells

Complete source mappings are deduplicated into canonical six-feature GrammarCells while retaining explicit source→cell relations.

In [6]:
canonical_raw = read_jsonl('canonical/cells.jsonl')
canonical_cells = pd.concat([
    canonical_raw[['cell_id']].reset_index(drop=True),
    pd.json_normalize(canonical_raw['features']).reset_index(drop=True),
], axis=1)
canonical_cells['source_support'] = canonical_raw['source_ids'].map(len).to_numpy()
canonical_cells['source_ids'] = canonical_raw['source_ids'].map(compact).to_numpy()
source_cell_relations = read_jsonl('canonical/source_cell_relations.jsonl')
selected_bank_for_support = read_jsonl('items/selected_bank.jsonl')
feature_columns = ['tense', 'aspect', 'voice', 'polarity', 'clause', 'modal']

cell_long = canonical_cells[['cell_id', *feature_columns]].melt(id_vars='cell_id', var_name='dimension', value_name='value')
relation_long = source_cell_relations.merge(canonical_cells[['cell_id', *feature_columns]], on='cell_id').melt(
    id_vars=['source_id', 'cell_id'], value_vars=feature_columns, var_name='dimension', value_name='value'
)
item_long = selected_bank_for_support[['item_id', 'cell_id']].merge(canonical_cells[['cell_id', *feature_columns]], on='cell_id').melt(
    id_vars=['item_id', 'cell_id'], value_vars=feature_columns, var_name='dimension', value_name='value'
)
feature_support = cell_long.groupby(['dimension', 'value']).agg(cells=('cell_id', 'nunique')).reset_index()
feature_support = feature_support.merge(
    relation_long.groupby(['dimension', 'value']).size().rename('source_cell_relations').reset_index(),
    on=['dimension', 'value'], how='left',
).merge(
    item_long.groupby(['dimension', 'value']).size().rename('selected_items').reset_index(),
    on=['dimension', 'value'], how='left',
)
canonical_compression = pd.DataFrame([{
    'complete_descriptors': int(mapping_rows['result'].eq('complete').sum()),
    'accepted_source_cell_relations': len(source_cell_relations),
    'unique_GrammarCells': len(canonical_cells),
    'complete_descriptors_per_cell': mapping_rows['result'].eq('complete').sum() / len(canonical_cells),
    'source_cell_relations_per_cell': len(source_cell_relations) / len(canonical_cells),
}])
show_table('Canonicalisation compression', canonical_compression)
show_table('Canonical GrammarCell inventory', canonical_cells)
show_table('Canonical feature/value support', feature_support.sort_values(['dimension', 'cells', 'value'], ascending=[True, False, True]))
show_table('Retained source→cell relations', source_cell_relations)

**Canonicalisation compression** — 1 row(s)

,complete_descriptors,accepted_source_cell_relations,unique_GrammarCells,complete_descriptors_per_cell,source_cell_relations_per_cell
0,44,48,24,1.833333,2.0


**Canonical GrammarCell inventory** — 24 row(s)

,cell_id,tense,aspect,voice,polarity,clause,modal,source_support,source_ids
0,cell_001,present,none,active,positive,polar_question,none,1,"[""1741163708336x930002080700909000""]"
1,cell_002,past,none,active,positive,polar_question,none,1,"[""1741163708336x930002080700909000""]"
2,cell_003,present,none,active,positive,declarative,none,4,"[""1741163708330x678606711578176000"", ""1741163713626x133661684035994100"", ""1741163713626x414037782388433340"", ""174116..."
3,cell_004,past,none,active,positive,declarative,none,4,"[""1741163708330x678606711578176000"", ""1741163712739x218459335733744320"", ""1741163712739x256584265013129380"", ""174116..."
4,cell_005,NA,none,active,positive,imperative,none,7,"[""1741163708330x197590942940390140"", ""1741163708332x482855077867024000"", ""1741163708331x843997098925344500"", ""174116..."
5,cell_006,NA,none,active,negative,imperative,none,4,"[""1741163716068x445742738976088800"", ""1741163708332x403656410301087360"", ""1741163708331x357890058176217500"", ""174116..."
6,cell_007,present,none,active,negative,declarative,none,2,"[""1741163713626x335670389512142900"", ""1741163716067x101148462288360720""]"
7,cell_008,present,progressive,active,positive,declarative,none,1,"[""1741163713121x244049823776565150""]"
8,cell_009,present,progressive,active,negative,declarative,none,2,"[""1741163713121x303053781636601900"", ""1741163716068x574314163168649300""]"
9,cell_010,past,none,active,negative,declarative,none,1,"[""1741163712739x998605611367738000""]"


**Canonical feature/value support** — 16 row(s)

,dimension,value,cells,source_cell_relations,selected_items
0,aspect,none,12,30,23
1,aspect,perfect,5,10,8
2,aspect,progressive,4,5,8
3,aspect,perfect_progressive,3,3,5
4,clause,declarative,20,35,37
5,clause,imperative,2,11,3
6,clause,polar_question,2,2,4
7,modal,none,23,47,42
8,modal,would,1,1,2
9,polarity,positive,16,31,28


**Retained source→cell relations** — 48 row(s)

,source_id,source_cell_index,cell_id,normalisation_note
0,1741163708336x930002080700909000,0,cell_001,phase2 eligible: tense
1,1741163708336x930002080700909000,1,cell_002,phase2 eligible: tense
2,1741163708330x678606711578176000,0,cell_003,phase2 eligible: tense
3,1741163708330x678606711578176000,1,cell_004,phase2 eligible: tense
4,1741163708330x197590942940390140,0,cell_005,None
5,1741163716068x445742738976088800,0,cell_006,None
6,1741163708332x482855077867024000,0,cell_005,source realization condition: emphatic-DO
7,1741163708331x843997098925344500,0,cell_005,source realization condition: LET'S
8,1741163708332x403656410301087360,0,cell_006,source realization condition: LET'S NOT
9,1741163708335x133935596064802990,0,cell_005,source realization condition: LET + third-person pronoun


,source_id,source_cell_index,cell_id,normalisation_note
0,1741163708336x930002080700909000,0,cell_001,phase2 eligible: tense
1,1741163708336x930002080700909000,1,cell_002,phase2 eligible: tense
2,1741163708330x678606711578176000,0,cell_003,phase2 eligible: tense
3,1741163708330x678606711578176000,1,cell_004,phase2 eligible: tense
4,1741163708330x197590942940390140,0,cell_005,None
5,1741163716068x445742738976088800,0,cell_006,None
6,1741163708332x482855077867024000,0,cell_005,source realization condition: emphatic-DO
7,1741163708331x843997098925344500,0,cell_005,source realization condition: LET'S
8,1741163708332x403656410301087360,0,cell_006,source realization condition: LET'S NOT
9,1741163708335x133935596064802990,0,cell_005,source realization condition: LET + third-person pronoun


## 4. Realistic item generation

Generation uses the fixed GrammarCell and item-design declarations only. Candidate indices 1–3 are the default best-of-three design; indices 4–5 are the preregistered zero-coverage rescue; indices 6–7 are the separate determinacy intervention.

In [7]:
generation_attempts_raw = read_jsonl('items/generation_attempts.jsonl')
generation_attempts = pd.json_normalize(generation_attempts_raw.to_dict('records'), sep='.')
generation_attempts['cohort'] = generation_attempts['candidate_index'].map(
    lambda index: 'default_N3' if index <= 3 else ('conditional_rescue' if index <= 5 else 'determinacy_intervention')
)
generation_attempts['structural_errors'] = generation_attempts['structural_errors'].map(compact)
candidate_rows = read_jsonl('items/candidates.jsonl')
candidate_table = pd.json_normalize(candidate_rows.to_dict('records'), sep='.')
candidate_table['accepted_answers'] = candidate_table['accepted_answers'].map(compact)
generation_funnel = generation_attempts.groupby('cohort').agg(
    attempts=('candidate_id', 'size'),
    cells=('cell_id', 'nunique'),
    structurally_valid=('structurally_valid', 'sum'),
    call_errors=('call_error', lambda values: values.notna().sum()),
    runtime_seconds=('runtime_seconds', 'sum'),
).reset_index()
generation_funnel['structural_validity_rate'] = generation_funnel['structurally_valid'] / generation_funnel['attempts']
attempt_columns = [
    'candidate_id', 'cell_id', 'candidate_index', 'cohort', 'structurally_valid',
    'structural_errors', 'call_error', 'runtime_seconds', 'model', 'provenance.status',
]
candidate_columns = [
    'item_id', 'cell_id', 'generation_metadata.candidate_index', 'format', 'prompt',
    'target_answer', 'accepted_answers', 'generation_metadata.model', 'generation_metadata.provenance.status',
]
show_table('Generation funnel by declared cohort', generation_funnel)
show_table('Generation attempts', generation_attempts[[column for column in attempt_columns if column in generation_attempts]])
show_table('Structurally valid generated candidates', candidate_table[[column for column in candidate_columns if column in candidate_table]])

**Generation funnel by declared cohort** — 3 row(s)

,cohort,attempts,cells,structurally_valid,call_errors,runtime_seconds,structural_validity_rate
0,conditional_rescue,4,2,4,0,54.835103,1.000000
1,default_N3,72,24,71,0,701.681103,0.986111
2,determinacy_intervention,2,1,2,0,27.638386,1.000000


**Generation attempts** — 78 row(s)

,candidate_id,cell_id,candidate_index,cohort,structurally_valid,structural_errors,call_error,runtime_seconds,model,provenance.status
0,candidate_cell_001_01,cell_001,1,default_N3,True,[],NaN,13.810211,gpt-5.6-sol,reused_live_model_evidence
1,candidate_cell_001_02,cell_001,2,default_N3,True,[],NaN,10.712408,gpt-5.6-sol,reused_live_model_evidence
2,candidate_cell_001_03,cell_001,3,default_N3,False,"[""note must be a non-empty string""]",NaN,15.384500,gpt-5.6-sol,reused_live_model_evidence
3,candidate_cell_002_01,cell_002,1,default_N3,True,[],NaN,8.160050,gpt-5.6-sol,phase6_live_model_evidence
4,candidate_cell_002_02,cell_002,2,default_N3,True,[],NaN,9.300156,gpt-5.6-sol,phase6_live_model_evidence
...,...,...,...,...,...,...,...,...,...,...
73,candidate_cell_023_02,cell_023,2,default_N3,True,[],NaN,6.591508,gpt-5.6-sol,phase6_live_model_evidence
74,candidate_cell_023_03,cell_023,3,default_N3,True,[],NaN,8.398603,gpt-5.6-sol,phase6_live_model_evidence
75,candidate_cell_024_01,cell_024,1,default_N3,True,[],NaN,13.463962,gpt-5.6-sol,reused_live_model_evidence
76,candidate_cell_024_02,cell_024,2,default_N3,True,[],NaN,14.510622,gpt-5.6-sol,reused_live_model_evidence


**Structurally valid generated candidates** — 77 row(s)

,item_id,cell_id,generation_metadata.candidate_index,format,prompt,target_answer,accepted_answers,generation_metadata.model,generation_metadata.provenance.status
0,candidate_cell_001_01,cell_001,1,controlled_production,You want to know whether Mia feeds the cat every morning. Use: Mia / feed the cat every morning. Response: ______,Does Mia feed the cat every morning?,"[""Does Mia feed the cat every morning?""]",gpt-5.6-sol,reused_live_model_evidence
1,candidate_cell_001_02,cell_001,2,controlled_production,Lena wants to know about Tom's daily trip to school. Complete the question using these words: Tom / walk / to school...,Does Tom walk to school every day?,"[""Does Tom walk to school every day?""]",gpt-5.6-sol,reused_live_model_evidence
2,candidate_cell_002_01,cell_002,1,controlled_production,You want to know about Mia's visit yesterday. Complete the question using “call her sister”: [_____],Did Mia call her sister?,"[""Did Mia call her sister?""]",gpt-5.6-sol,NaN
3,candidate_cell_002_02,cell_002,2,controlled_production,You want to know whether Mia took the bus yesterday. Complete the question using the cue “take the bus”: [____]?,Did Mia take the bus yesterday?,"[""Did Mia take the bus yesterday""]",gpt-5.6-sol,NaN
4,candidate_cell_002_03,cell_002,3,controlled_production,You cannot remember whether Mia opened the window yesterday. Ask: ____?,Did Mia open the window yesterday?,"[""Did Mia open the window yesterday""]",gpt-5.6-sol,NaN
...,...,...,...,...,...,...,...,...,...
72,candidate_cell_023_02,cell_023,2,controlled_production,"The café serves lunch, but it closes before dinner. Complete the sentence using “dinner” and “serve”: Dinner ____.",Dinner is not served.,"[""is not served"", ""isn't served""]",gpt-5.6-sol,NaN
73,candidate_cell_023_03,cell_023,3,controlled_production,"Every morning, Mia washes the red cups but leaves the blue cup alone. Complete the sentence using “wash”: The blue c...",The blue cup is not washed.,"[""is not washed"", ""isn't washed""]",gpt-5.6-sol,NaN
74,candidate_cell_024_01,cell_024,1,controlled_production,Ben offered to help with the move. Use the cue “carry the boxes” to complete his response: ____.,He would carry the boxes.,"[""He would carry the boxes.""]",gpt-5.6-sol,reused_live_model_evidence
75,candidate_cell_024_02,cell_024,2,controlled_production,The car is unavailable today. Complete the response using the words in brackets.\nA: How would Ben get to work?\nB: ...,He would take the bus.,"[""He would take the bus."", ""He'd take the bus.""]",gpt-5.6-sol,reused_live_model_evidence


,item_id,cell_id,generation_metadata.candidate_index,format,prompt,target_answer,accepted_answers,generation_metadata.model,generation_metadata.provenance.status
0,candidate_cell_001_01,cell_001,1,controlled_production,You want to know whether Mia feeds the cat every morning. Use: Mia / feed the cat every morning. Response: ______,Does Mia feed the cat every morning?,"[""Does Mia feed the cat every morning?""]",gpt-5.6-sol,reused_live_model_evidence
1,candidate_cell_001_02,cell_001,2,controlled_production,Lena wants to know about Tom's daily trip to school. Complete the question using these words: Tom / walk / to school...,Does Tom walk to school every day?,"[""Does Tom walk to school every day?""]",gpt-5.6-sol,reused_live_model_evidence
2,candidate_cell_002_01,cell_002,1,controlled_production,You want to know about Mia's visit yesterday. Complete the question using “call her sister”: [_____],Did Mia call her sister?,"[""Did Mia call her sister?""]",gpt-5.6-sol,NaN
3,candidate_cell_002_02,cell_002,2,controlled_production,You want to know whether Mia took the bus yesterday. Complete the question using the cue “take the bus”: [____]?,Did Mia take the bus yesterday?,"[""Did Mia take the bus yesterday""]",gpt-5.6-sol,NaN
4,candidate_cell_002_03,cell_002,3,controlled_production,You cannot remember whether Mia opened the window yesterday. Ask: ____?,Did Mia open the window yesterday?,"[""Did Mia open the window yesterday""]",gpt-5.6-sol,NaN
...,...,...,...,...,...,...,...,...,...
72,candidate_cell_023_02,cell_023,2,controlled_production,"The café serves lunch, but it closes before dinner. Complete the sentence using “dinner” and “serve”: Dinner ____.",Dinner is not served.,"[""is not served"", ""isn't served""]",gpt-5.6-sol,NaN
73,candidate_cell_023_03,cell_023,3,controlled_production,"Every morning, Mia washes the red cups but leaves the blue cup alone. Complete the sentence using “wash”: The blue c...",The blue cup is not washed.,"[""is not washed"", ""isn't washed""]",gpt-5.6-sol,NaN
74,candidate_cell_024_01,cell_024,1,controlled_production,Ben offered to help with the move. Use the cue “carry the boxes” to complete his response: ____.,He would carry the boxes.,"[""He would carry the boxes.""]",gpt-5.6-sol,reused_live_model_evidence
75,candidate_cell_024_02,cell_024,2,controlled_production,The car is unavailable today. Complete the response using the words in brackets.\nA: How would Ben get to work?\nB: ...,He would take the bus.,"[""He would take the bus."", ""He'd take the bus.""]",gpt-5.6-sol,reused_live_model_evidence


## 5. Independent validation, curation, and fixed item bank

The active tables use the curated candidate packages and independently retained curated judgments. Raw generation and validation remain immutable audit evidence. Six answer-key packages were corrected and independently rejudged without changing prompts, cell assignments, or item IDs.

In [8]:
raw_validation = read_jsonl('items/validation.jsonl')
curated_validation_raw = read_jsonl('items/curated_validation.jsonl')
curated_validation = pd.json_normalize(curated_validation_raw.to_dict('records'), sep='.')
curated_candidates_raw = read_jsonl('items/curated_candidates.jsonl')
curated_candidates = pd.json_normalize(curated_candidates_raw.to_dict('records'), sep='.')
criterion_columns = sorted(column for column in curated_validation if column.startswith('judgments.') and column.endswith('.passed'))
criterion_names = [column.removeprefix('judgments.').removesuffix('.passed') for column in criterion_columns]
validation_decisions = curated_validation[['item_id', 'accepted', 'rejection_stage', *criterion_columns]].copy()
validation_decisions.columns = ['item_id', 'accepted', 'rejection_stage', *criterion_names]
criterion_results = pd.DataFrame([
    {
        'criterion': criterion,
        'passed': int(curated_validation[column].fillna(False).sum()),
        'failed': int((~curated_validation[column].fillna(False)).sum()),
        'pass_rate': curated_validation[column].fillna(False).mean(),
    }
    for criterion, column in zip(criterion_names, criterion_columns)
])
rejection_summary = curated_validation.assign(
    decision=curated_validation['accepted'].map({True: 'accepted', False: 'rejected'}),
    rejection_stage=curated_validation['rejection_stage'].fillna('accepted'),
).groupby(['decision', 'rejection_stage']).size().rename('items').reset_index()

corrected_items = curated_candidates.loc[curated_candidates['curation_metadata.corrected'].fillna(False)].merge(
    curated_validation[['item_id', 'accepted', 'validation_metadata.status']], on='item_id', how='left'
)
correction_columns = [
    'item_id', 'cell_id', 'prompt', 'target_answer', 'accepted_answers',
    'curation_metadata.corrected_fields', 'accepted', 'validation_metadata.status',
]
for column in ('accepted_answers', 'curation_metadata.corrected_fields'):
    if column in corrected_items:
        corrected_items[column] = corrected_items[column].map(compact)

selected_bank_raw = read_jsonl('items/selected_bank.jsonl')
selected_bank = pd.json_normalize(selected_bank_raw.to_dict('records'), sep='.')
bank_columns = [
    'item_id', 'cell_id', 'selection_metadata.rank', 'selection_metadata.rule',
    'generation_metadata.candidate_index', 'curation_metadata.corrected',
    'format', 'prompt', 'target_answer', 'accepted_answers',
]
selected_bank_display = selected_bank[[column for column in bank_columns if column in selected_bank]].copy()
selected_bank_display['accepted_answers'] = selected_bank_display['accepted_answers'].map(compact)
bank_summary_json = read_json('items/bank_summary.json')
bank_headline_keys = [
    'generated_candidates', 'validator_accepted_candidates', 'validator_acceptance_rate',
    'selected_bank_items', 'selection_rate_among_accepted', 'grammar_cells',
    'covered_cells', 'unique_prompt_rate', 'lexical_types', 'lexical_tokens', 'lexical_diversity',
]
bank_headline = pd.DataFrame([{'metric': key, 'value': bank_summary_json[key]} for key in bank_headline_keys])
audit_counts = pd.DataFrame([{
    'raw_candidates': len(candidate_rows),
    'curated_candidates': len(curated_candidates),
    'raw_judgments': len(raw_validation),
    'curated_judgments': len(curated_validation),
    'packaging_corrections': int(curated_candidates['curation_metadata.corrected'].fillna(False).sum()),
    'corrected_candidates_accepted': int(corrected_items['accepted'].fillna(False).sum()),
    'fixed_bank_items': len(selected_bank),
}])
show_table('Validation and fixed-bank headline results', bank_headline)
show_table('Raw-to-curated audit counts', audit_counts)
show_table('Validation criteria', criterion_results.sort_values(['pass_rate', 'criterion']))
show_table('Acceptance and rejection stages', rejection_summary)
show_table('Per-item active validation decisions', validation_decisions)
show_table('Frozen packaging corrections', corrected_items[[column for column in correction_columns if column in corrected_items]])
show_table('Fixed learner-facing item bank', selected_bank_display)

**Validation and fixed-bank headline results** — 11 row(s)

,metric,value
0,generated_candidates,77.000000
1,validator_accepted_candidates,54.000000
2,validator_acceptance_rate,0.701299
3,selected_bank_items,44.000000
4,selection_rate_among_accepted,0.814815
5,grammar_cells,24.000000
6,covered_cells,24.000000
7,unique_prompt_rate,1.000000
8,lexical_types,242.000000
9,lexical_tokens,795.000000


**Raw-to-curated audit counts** — 1 row(s)

,raw_candidates,curated_candidates,raw_judgments,curated_judgments,packaging_corrections,corrected_candidates_accepted,fixed_bank_items
0,77,77,77,77,6,4,44


**Validation criteria** — 9 row(s)

...

**Frozen packaging corrections** — 6 row(s)

,item_id,cell_id,prompt,target_answer,accepted_answers,curation_metadata.corrected_fields,accepted,validation_metadata.status
0,candidate_cell_003_01,cell_003,"Emma has a dog. Every morning, she ___ before breakfast. (walk / the dog)","Every morning, she walks the dog before breakfast.","[""walks the dog""]","[""target_answer""]",True,phase6_frozen_item_packaging_correction
1,candidate_cell_005_01,cell_005,The room is dark. Tell your friend what to do using the cue “turn on / light”: _____.,Turn on the light.,"[""Turn on the light""]","[""accepted_answers""]",False,phase6_frozen_item_packaging_correction
2,candidate_cell_017_06,cell_017,Nina was tired because she [____] well for several nights. Use the negative past perfect continuous form of “sleep.”,Nina was tired because she had not been sleeping well for several nights.,"[""had not been sleeping"", ""hadn't been sleeping""]","[""target_answer""]",True,phase6_frozen_item_packaging_correction
3,candidate_cell_018_01,cell_018,"Maya started at eight. At noon, she was still doing the same job. Use the cue “paint the fence” to complete the sent...",She had been painting the fence for four hours.,"[""She had been painting the fence for four hours"", ""Maya had been painting the fence for four hours""]","[""accepted_answers""]",False,phase6_frozen_item_packaging_correction
4,candidate_cell_018_03,cell_018,"Maya started painting the fence at 8:00. At noon, she was still working. Complete the sentence using the cue (Maya /...","At noon, Maya had been painting the fence for four hours.","[""Maya had been painting the fence for four hours""]","[""accepted_answers"", ""target_answer""]",True,phase6_frozen_item_packaging_correction
5,candidate_cell_019_03,cell_019,"Yesterday, a worker painted the fence. Complete the sentence using the cue (the fence / paint): _____.",The fence was painted.,"[""The fence was painted""]","[""accepted_answers""]",True,phase6_frozen_item_packaging_correction


**Fixed learner-facing item bank** — 44 row(s)

,item_id,cell_id,selection_metadata.rank,selection_metadata.rule,generation_metadata.candidate_index,curation_metadata.corrected,format,prompt,target_answer,accepted_answers
0,candidate_cell_001_01,cell_001,1,earliest_valid,1,False,controlled_production,You want to know whether Mia feeds the cat every morning. Use: Mia / feed the cat every morning. Response: ______,Does Mia feed the cat every morning?,"[""Does Mia feed the cat every morning?""]"
1,candidate_cell_001_02,cell_001,2,maximum_token_set_distance_from_first,2,False,controlled_production,Lena wants to know about Tom's daily trip to school. Complete the question using these words: Tom / walk / to school...,Does Tom walk to school every day?,"[""Does Tom walk to school every day?""]"
2,candidate_cell_002_01,cell_002,1,earliest_valid,1,False,controlled_production,You want to know about Mia's visit yesterday. Complete the question using “call her sister”: [_____],Did Mia call her sister?,"[""Did Mia call her sister?""]"
3,candidate_cell_002_03,cell_002,2,maximum_token_set_distance_from_first,3,False,controlled_production,You cannot remember whether Mia opened the window yesterday. Ask: ____?,Did Mia open the window yesterday?,"[""Did Mia open the window yesterday""]"
4,candidate_cell_003_01,cell_003,1,earliest_valid,1,True,controlled_production,"Emma has a dog. Every morning, she ___ before breakfast. (walk / the dog)","Every morning, she walks the dog before breakfast.","[""walks the dog""]"
5,candidate_cell_003_03,cell_003,2,maximum_token_set_distance_from_first,3,False,controlled_production,Maya has the same routine every morning. Complete the sentence using “pack”: Maya ____ her lunch before school.,Maya packs her lunch before school.,"[""packs""]"
6,candidate_cell_004_01,cell_004,1,earliest_valid,1,False,controlled_production,"Yesterday, Mia found a dirty cup on the table. Complete the sentence using “wash”: Mia ____ the cup.",Mia washed the cup.,"[""washed""]"
7,candidate_cell_004_02,cell_004,2,maximum_token_set_distance_from_first,2,False,controlled_production,"Yesterday, Leo needed some fruit for breakfast. Complete the sentence using “buy”: Leo ____ some apples at the shop.",Leo bought some apples at the shop.,"[""bought""]"
8,candidate_cell_005_03,cell_005,1,earliest_valid,3,False,controlled_production,The room is dark. Tell your friend what to do. Use the cue: turn on / the light\n[____],Turn on the light.,"[""Turn on the light.""]"
9,candidate_cell_006_01,cell_006,1,earliest_valid,1,False,controlled_production,The pan is very hot. Warn a child. Use the cue TOUCH: ____,Don't touch the pan.,"[""Don't touch the pan."", ""Do not touch the pan.""]"


,item_id,cell_id,selection_metadata.rank,selection_metadata.rule,generation_metadata.candidate_index,curation_metadata.corrected,format,prompt,target_answer,accepted_answers
0,candidate_cell_001_01,cell_001,1,earliest_valid,1,False,controlled_production,You want to know whether Mia feeds the cat every morning. Use: Mia / feed the cat every morning. Response: ______,Does Mia feed the cat every morning?,"[""Does Mia feed the cat every morning?""]"
1,candidate_cell_001_02,cell_001,2,maximum_token_set_distance_from_first,2,False,controlled_production,Lena wants to know about Tom's daily trip to school. Complete the question using these words: Tom / walk / to school...,Does Tom walk to school every day?,"[""Does Tom walk to school every day?""]"
2,candidate_cell_002_01,cell_002,1,earliest_valid,1,False,controlled_production,You want to know about Mia's visit yesterday. Complete the question using “call her sister”: [_____],Did Mia call her sister?,"[""Did Mia call her sister?""]"
3,candidate_cell_002_03,cell_002,2,maximum_token_set_distance_from_first,3,False,controlled_production,You cannot remember whether Mia opened the window yesterday. Ask: ____?,Did Mia open the window yesterday?,"[""Did Mia open the window yesterday""]"
4,candidate_cell_003_01,cell_003,1,earliest_valid,1,True,controlled_production,"Emma has a dog. Every morning, she ___ before breakfast. (walk / the dog)","Every morning, she walks the dog before breakfast.","[""walks the dog""]"
5,candidate_cell_003_03,cell_003,2,maximum_token_set_distance_from_first,3,False,controlled_production,Maya has the same routine every morning. Complete the sentence using “pack”: Maya ____ her lunch before school.,Maya packs her lunch before school.,"[""packs""]"
6,candidate_cell_004_01,cell_004,1,earliest_valid,1,False,controlled_production,"Yesterday, Mia found a dirty cup on the table. Complete the sentence using “wash”: Mia ____ the cup.",Mia washed the cup.,"[""washed""]"
7,candidate_cell_004_02,cell_004,2,maximum_token_set_distance_from_first,2,False,controlled_production,"Yesterday, Leo needed some fruit for breakfast. Complete the sentence using “buy”: Leo ____ some apples at the shop.",Leo bought some apples at the shop.,"[""bought""]"
8,candidate_cell_005_03,cell_005,1,earliest_valid,3,False,controlled_production,The room is dark. Tell your friend what to do. Use the cue: turn on / the light\n[____],Turn on the light.,"[""Turn on the light.""]"
9,candidate_cell_006_01,cell_006,1,earliest_valid,1,False,controlled_production,The pan is very hot. Warn a child. Use the cue TOUCH: ____,Don't touch the pan.,"[""Don't touch the pan."", ""Do not touch the pan.""]"


## 6. Outcome-free semantic grammar fold

Development cells support candidate construction and acquisition. Compositional cells contain seen constituents in unseen exact tuples; novel-feature cells contain at least one value unseen in development. Outcomes are not used to construct this fold.

In [9]:
fold_raw = read_jsonl('fold/assignments.jsonl')
fold_assignments = pd.concat([
    fold_raw.drop(columns=['features']).reset_index(drop=True),
    pd.json_normalize(fold_raw['features']).reset_index(drop=True),
], axis=1)
fold_assignments['accepted_item_ids'] = fold_assignments['accepted_item_ids'].map(compact)
fold_assignments['unseen_development_values'] = fold_assignments['unseen_development_values'].map(compact)
fold_summary = fold_assignments.groupby('grammar_split').agg(
    cells=('cell_id', 'nunique'),
    selected_items=('accepted_item_support', 'sum'),
    minimum_items_per_cell=('accepted_item_support', 'min'),
    maximum_items_per_cell=('accepted_item_support', 'max'),
).reset_index()
bank_with_grammar = selected_bank_display.merge(
    fold_assignments[['cell_id', 'grammar_split', *feature_columns]], on='cell_id', how='left'
)
show_table('Grammar-fold summary', fold_summary)
show_table('Semantic GrammarCell assignments', fold_assignments)
show_table('Final item bank with canonical features and grammar regime', bank_with_grammar)

**Grammar-fold summary** — 3 row(s)

,grammar_split,cells,selected_items,minimum_items_per_cell,maximum_items_per_cell
0,compositional_holdout,5,10,2,2
1,development,18,32,1,2
2,novel_feature_holdout,1,2,2,2


**Semantic GrammarCell assignments** — 24 row(s)

,cell_id,grammar_split,accepted_item_ids,accepted_item_support,unseen_development_values,selection_reason,tense,aspect,voice,polarity,clause,modal
0,cell_020,compositional_holdout,"[""candidate_cell_020_01"", ""candidate_cell_020_03""]",2,[],semantic_sample_with_supported_development_constituents,present,none,passive,positive,declarative,none
1,cell_007,compositional_holdout,"[""candidate_cell_007_01"", ""candidate_cell_007_02""]",2,[],semantic_sample_with_supported_development_constituents,present,none,active,negative,declarative,none
2,cell_011,compositional_holdout,"[""candidate_cell_011_01"", ""candidate_cell_011_03""]",2,[],semantic_sample_with_supported_development_constituents,past,progressive,active,positive,declarative,none
3,cell_019,compositional_holdout,"[""candidate_cell_019_02"", ""candidate_cell_019_03""]",2,[],semantic_sample_with_supported_development_constituents,past,none,passive,positive,declarative,none
4,cell_010,compositional_holdout,"[""candidate_cell_010_01"", ""candidate_cell_010_03""]",2,[],semantic_sample_with_supported_development_constituents,past,none,active,negative,declarative,none
5,cell_003,development,"[""candidate_cell_003_01"", ""candidate_cell_003_03""]",2,[],development_acquisition,present,none,active,positive,declarative,none
6,cell_018,development,"[""candidate_cell_018_03""]",1,[],development_acquisition,past,perfect_progressive,active,positive,declarative,none
7,cell_017,development,"[""candidate_cell_017_06"", ""candidate_cell_017_07""]",2,[],development_acquisition,past,perfect_progressive,active,negative,declarative,none
8,cell_002,development,"[""candidate_cell_002_01"", ""candidate_cell_002_03""]",2,[],development_acquisition,past,none,active,positive,polar_question,none
9,cell_005,development,"[""candidate_cell_005_03""]",1,[],development_acquisition,NA,none,active,positive,imperative,none


**Final item bank with canonical features and grammar regime** — 44 row(s)

,item_id,cell_id,selection_metadata.rank,selection_metadata.rule,generation_metadata.candidate_index,curation_metadata.corrected,format,prompt,target_answer,accepted_answers,grammar_split,tense,aspect,voice,polarity,clause,modal
0,candidate_cell_001_01,cell_001,1,earliest_valid,1,False,controlled_production,You want to know whether Mia feeds the cat every morning. Use: Mia / feed the cat every morning. Response: ______,Does Mia feed the cat every morning?,"[""Does Mia feed the cat every morning?""]",development,present,none,active,positive,polar_question,none
1,candidate_cell_001_02,cell_001,2,maximum_token_set_distance_from_first,2,False,controlled_production,Lena wants to know about Tom's daily trip to school. Complete the question using these words: Tom / walk / to school...,Does Tom walk to school every day?,"[""Does Tom walk to school every day?""]",development,present,none,active,positive,polar_question,none
2,candidate_cell_002_01,cell_002,1,earliest_valid,1,False,controlled_production,You want to know about Mia's visit yesterday. Complete the question using “call her sister”: [_____],Did Mia call her sister?,"[""Did Mia call her sister?""]",development,past,none,active,positive,polar_question,none
3,candidate_cell_002_03,cell_002,2,maximum_token_set_distance_from_first,3,False,controlled_production,You cannot remember whether Mia opened the window yesterday. Ask: ____?,Did Mia open the window yesterday?,"[""Did Mia open the window yesterday""]",development,past,none,active,positive,polar_question,none
4,candidate_cell_003_01,cell_003,1,earliest_valid,1,True,controlled_production,"Emma has a dog. Every morning, she ___ before breakfast. (walk / the dog)","Every morning, she walks the dog before breakfast.","[""walks the dog""]",development,present,none,active,positive,declarative,none
5,candidate_cell_003_03,cell_003,2,maximum_token_set_distance_from_first,3,False,controlled_production,Maya has the same routine every morning. Complete the sentence using “pack”: Maya ____ her lunch before school.,Maya packs her lunch before school.,"[""packs""]",development,present,none,active,positive,declarative,none
6,candidate_cell_004_01,cell_004,1,earliest_valid,1,False,controlled_production,"Yesterday, Mia found a dirty cup on the table. Complete the sentence using “wash”: Mia ____ the cup.",Mia washed the cup.,"[""washed""]",development,past,none,active,positive,declarative,none
7,candidate_cell_004_02,cell_004,2,maximum_token_set_distance_from_first,2,False,controlled_production,"Yesterday, Leo needed some fruit for breakfast. Complete the sentence using “buy”: Leo ____ some apples at the shop.",Leo bought some apples at the shop.,"[""bought""]",development,past,none,active,positive,declarative,none
8,candidate_cell_005_03,cell_005,1,earliest_valid,3,False,controlled_production,The room is dark. Tell your friend what to do. Use the cue: turn on / the light\n[____],Turn on the light.,"[""Turn on the light.""]",development,NA,none,active,positive,imperative,none
9,candidate_cell_006_01,cell_006,1,earliest_valid,1,False,controlled_production,The pan is very hot. Warn a child. Use the cue TOUCH: ____,Don't touch the pan.,"[""Don't touch the pan."", ""Do not touch the pan.""]",development,NA,none,active,negative,imperative,none


,item_id,cell_id,selection_metadata.rank,selection_metadata.rule,generation_metadata.candidate_index,curation_metadata.corrected,format,prompt,target_answer,accepted_answers,grammar_split,tense,aspect,voice,polarity,clause,modal
0,candidate_cell_001_01,cell_001,1,earliest_valid,1,False,controlled_production,You want to know whether Mia feeds the cat every morning. Use: Mia / feed the cat every morning. Response: ______,Does Mia feed the cat every morning?,"[""Does Mia feed the cat every morning?""]",development,present,none,active,positive,polar_question,none
1,candidate_cell_001_02,cell_001,2,maximum_token_set_distance_from_first,2,False,controlled_production,Lena wants to know about Tom's daily trip to school. Complete the question using these words: Tom / walk / to school...,Does Tom walk to school every day?,"[""Does Tom walk to school every day?""]",development,present,none,active,positive,polar_question,none
2,candidate_cell_002_01,cell_002,1,earliest_valid,1,False,controlled_production,You want to know about Mia's visit yesterday. Complete the question using “call her sister”: [_____],Did Mia call her sister?,"[""Did Mia call her sister?""]",development,past,none,active,positive,polar_question,none
3,candidate_cell_002_03,cell_002,2,maximum_token_set_distance_from_first,3,False,controlled_production,You cannot remember whether Mia opened the window yesterday. Ask: ____?,Did Mia open the window yesterday?,"[""Did Mia open the window yesterday""]",development,past,none,active,positive,polar_question,none
4,candidate_cell_003_01,cell_003,1,earliest_valid,1,True,controlled_production,"Emma has a dog. Every morning, she ___ before breakfast. (walk / the dog)","Every morning, she walks the dog before breakfast.","[""walks the dog""]",development,present,none,active,positive,declarative,none
5,candidate_cell_003_03,cell_003,2,maximum_token_set_distance_from_first,3,False,controlled_production,Maya has the same routine every morning. Complete the sentence using “pack”: Maya ____ her lunch before school.,Maya packs her lunch before school.,"[""packs""]",development,present,none,active,positive,declarative,none
6,candidate_cell_004_01,cell_004,1,earliest_valid,1,False,controlled_production,"Yesterday, Mia found a dirty cup on the table. Complete the sentence using “wash”: Mia ____ the cup.",Mia washed the cup.,"[""washed""]",development,past,none,active,positive,declarative,none
7,candidate_cell_004_02,cell_004,2,maximum_token_set_distance_from_first,2,False,controlled_production,"Yesterday, Leo needed some fruit for breakfast. Complete the sentence using “buy”: Leo ____ some apples at the shop.",Leo bought some apples at the shop.,"[""bought""]",development,past,none,active,positive,declarative,none
8,candidate_cell_005_03,cell_005,1,earliest_valid,3,False,controlled_production,The room is dark. Tell your friend what to do. Use the cue: turn on / the light\n[____],Turn on the light.,"[""Turn on the light.""]",development,NA,none,active,positive,imperative,none
9,candidate_cell_006_01,cell_006,1,earliest_valid,1,False,controlled_production,The pan is very hot. Warn a child. Use the cue TOUCH: ____,Don't touch the pan.,"[""Don't touch the pan."", ""Do not touch the pan.""]",development,NA,none,active,negative,imperative,none


## 7. Development-derived KC candidate space

Feature-value, English-declared operation, supported pairwise-interaction, and exact-development-cell candidates are constructed without learner outcomes. Structural support and activation equivalence are measured on development items before selection.

In [10]:
candidate_inventory = read_json('kc/candidate_inventory.json')
kc_candidates_raw = pd.json_normalize(candidate_inventory['candidates'], sep='.')
kc_candidate_table = kc_candidates_raw[[
    'id', 'family', 'definition', 'cell_support', 'item_support', 'meets_support_threshold',
    'equivalence_class_id', 'equivalent_to', 'is_equivalence_representative',
    'selection_eligible', 'exclusion_reasons', 'supporting_development_cell_ids',
    'supporting_development_item_ids',
]].copy()
for column in ('exclusion_reasons', 'supporting_development_cell_ids', 'supporting_development_item_ids'):
    kc_candidate_table[column] = kc_candidate_table[column].map(compact)
candidate_family_summary = kc_candidates_raw.assign(
    activation_duplicate=kc_candidates_raw['equivalent_to'].notna(),
).groupby('family').agg(
    candidates=('id', 'size'),
    support_eligible=('meets_support_threshold', 'sum'),
    selection_eligible=('selection_eligible', 'sum'),
    activation_duplicates=('activation_duplicate', 'sum'),
    median_cell_support=('cell_support', 'median'),
    median_item_support=('item_support', 'median'),
    maximum_item_support=('item_support', 'max'),
).reset_index()
equivalence_classes = pd.DataFrame([
    {
        'equivalence_class_id': row['equivalence_class_id'],
        'representative_id': row['representative_id'],
        'member_count': len(row['member_ids']),
        'activation_item_support': sum(row['activation_vector']),
        'member_ids': compact(row['member_ids']),
    }
    for row in candidate_inventory['equivalence_classes']
])
candidate_count_table = pd.DataFrame([
    {'metric': metric, 'value': value}
    for metric, value in candidate_inventory['candidate_counts'].items()
])
support_threshold_table = pd.DataFrame([
    {'threshold': name, 'value': value}
    for name, value in candidate_inventory['support_thresholds'].items()
])
show_table('KC candidate-space counts', candidate_count_table)
show_table('Structural support thresholds', support_threshold_table)
show_table('Candidate families and structural support', candidate_family_summary)
show_table('All development-derived KC candidates', kc_candidate_table)
show_table('Development-item activation equivalence classes', equivalence_classes.sort_values(['member_count', 'equivalence_class_id'], ascending=[False, True]))

**KC candidate-space counts** — 9 row(s)

,metric,value
0,feature_value,9
1,operation,10
2,interaction,18
3,full_cell,18
4,raw_total,55
5,support_eligible,42
6,selection_eligible,28
7,activation_equivalence_classes,38
8,activation_duplicate_candidates,17


**Structural support thresholds** — 2 row(s)

,threshold,value
0,minimum_interaction_cell_support,2
1,minimum_interaction_item_support,3


**Candidate families and structural support** — 4 row(s)

...

**All development-derived KC candidates** — 55 row(s)

,id,family,definition,cell_support,item_support,meets_support_threshold,equivalence_class_id,equivalent_to,is_equivalence_representative,selection_eligible,exclusion_reasons,supporting_development_cell_ids,supporting_development_item_ids
0,kc_cell__tense_na__aspect_none__voice_active__polarity_negative__clause_imperative__modal_none,full_cell,"Represent the exact development GrammarCell tense=NA, aspect=none, voice=active, polarity=negative, clause=imperativ...",1,2,True,activation_class_027,kc_interaction__clause_imperative__and__polarity_negative,False,False,"[""activation_equivalent_on_development_items""]","[""cell_006""]","[""candidate_cell_006_01"", ""candidate_cell_006_03""]"
1,kc_cell__tense_na__aspect_none__voice_active__polarity_positive__clause_imperative__modal_none,full_cell,"Represent the exact development GrammarCell tense=NA, aspect=none, voice=active, polarity=positive, clause=imperativ...",1,1,True,activation_class_029,None,True,True,[],"[""cell_005""]","[""candidate_cell_005_03""]"
2,kc_cell__tense_past__aspect_none__voice_active__polarity_positive__clause_declarative__modal_none,full_cell,"Represent the exact development GrammarCell tense=past, aspect=none, voice=active, polarity=positive, clause=declara...",1,2,True,activation_class_031,None,True,True,[],"[""cell_004""]","[""candidate_cell_004_01"", ""candidate_cell_004_02""]"
3,kc_cell__tense_past__aspect_none__voice_active__polarity_positive__clause_polar_question__modal_none,full_cell,"Represent the exact development GrammarCell tense=past, aspect=none, voice=active, polarity=positive, clause=polar_q...",1,2,True,activation_class_033,kc_interaction__clause_polar_question__and__tense_past,False,False,"[""activation_equivalent_on_development_items""]","[""cell_002""]","[""candidate_cell_002_01"", ""candidate_cell_002_03""]"
4,kc_cell__tense_past__aspect_perfect__voice_active__polarity_negative__clause_declarative__modal_none,full_cell,"Represent the exact development GrammarCell tense=past, aspect=perfect, voice=active, polarity=negative, clause=decl...",1,2,True,activation_class_013,None,True,True,[],"[""cell_014""]","[""candidate_cell_014_01"", ""candidate_cell_014_03""]"
5,kc_cell__tense_past__aspect_perfect__voice_active__polarity_positive__clause_declarative__modal_none,full_cell,"Represent the exact development GrammarCell tense=past, aspect=perfect, voice=active, polarity=positive, clause=decl...",1,1,True,activation_class_012,None,True,True,[],"[""cell_015""]","[""candidate_cell_015_03""]"
6,kc_cell__tense_past__aspect_perfect__voice_passive__polarity_positive__clause_declarative__modal_none,full_cell,"Represent the exact development GrammarCell tense=past, aspect=perfect, voice=passive, polarity=positive, clause=dec...",1,1,True,activation_class_003,kc_interaction__aspect_perfect__and__voice_passive,False,False,"[""activation_equivalent_on_development_items""]","[""cell_022""]","[""candidate_cell_022_05""]"
7,kc_cell__tense_past__aspect_perfect_progressive__voice_active__polarity_negative__clause_declarative__modal_none,full_cell,"Represent the exact development GrammarCell tense=past, aspect=perfect_progressive, voice=active, polarity=negative,...",1,2,True,activation_class_008,kc_interaction__aspect_perfect_progressive__and__polarity_negative,False,False,"[""activation_equivalent_on_development_items""]","[""cell_017""]","[""candidate_cell_017_06"", ""candidate_cell_017_07""]"
8,kc_cell__tense_past__aspect_perfect_progressive__voice_active__polarity_positive__clause_declarative__modal_none,full_cell,"Represent the exact development GrammarCell tense=past, aspect=perfect_progressive, voice=active, polarity=positive,...",1,1,True,activation_class_007,None,True,True,[],"[""cell_018""]","[""candidate_cell_018_03""]"
9,kc_cell__tense_present__aspect_none__voice_active__polarity_positive__clause_declarative__modal_none,full_cell,"Represent the exact development GrammarCell tense=present, aspect=none, voice=active, polarity=positive, clause=decl...",1

**Development-item activation equivalence classes** — 38 row(s)

,equivalence_class_id,representative_id,member_count,activation_item_support,member_ids
0,activation_class_001,kc_operation__central_modal,3,0,"[""kc_operation__central_modal"", ""kc_operation__subject_wh"", ""kc_operation__wh_fronting""]"
1,activation_class_003,kc_interaction__aspect_perfect__and__voice_passive,3,1,"[""kc_interaction__aspect_perfect__and__voice_passive"", ""kc_interaction__tense_past__and__voice_passive"", ""kc_cell__t..."
2,activation_class_002,kc_interaction__polarity_negative__and__voice_passive,2,2,"[""kc_interaction__polarity_negative__and__voice_passive"", ""kc_cell__tense_present__aspect_none__voice_passive__polar..."
3,activation_class_004,kc_interaction__aspect_progressive__and__voice_passive,2,2,"[""kc_interaction__aspect_progressive__and__voice_passive"", ""kc_cell__tense_present__aspect_progressive__voice_passiv..."
4,activation_class_006,kc_feature__voice__passive,2,5,"[""kc_feature__voice__passive"", ""kc_operation__passive_dependency""]"
5,activation_class_008,kc_interaction__aspect_perfect_progressive__and__polarity_negative,2,2,"[""kc_interaction__aspect_perfect_progressive__and__polarity_negative"", ""kc_cell__tense_past__aspect_perfect_progress..."
6,activation_class_010,kc_interaction__aspect_perfect_progressive__and__tense_present,2,2,"[""kc_interaction__aspect_perfect_progressive__and__tense_present"", ""kc_cell__tense_present__aspect_perfect_progressi..."
7,activation_class_022,kc_interaction__aspect_progressive__and__polarity_negative,2,2,"[""kc_interaction__aspect_progressive__and__polarity_negative"", ""kc_cell__tense_present__aspect_progressive__voice_ac..."
8,activation_class_025,kc_feature__aspect__progressive,2,6,"[""kc_feature__aspect__progressive"", ""kc_interaction__aspect_progressive__and__tense_present""]"
9,activation_class_027,kc_interaction__clause_imperative__and__polarity_negative,2,2,"[""kc_interaction__clause_imperative__and__polarity_negative"", ""kc_cell__tense_na__aspect_none__voice_active__polarit..."


,equivalence_class_id,representative_id,member_count,activation_item_support,member_ids
0,activation_class_001,kc_operation__central_modal,3,0,"[""kc_operation__central_modal"", ""kc_operation__subject_wh"", ""kc_operation__wh_fronting""]"
1,activation_class_003,kc_interaction__aspect_perfect__and__voice_passive,3,1,"[""kc_interaction__aspect_perfect__and__voice_passive"", ""kc_interaction__tense_past__and__voice_passive"", ""kc_cell__t..."
2,activation_class_002,kc_interaction__polarity_negative__and__voice_passive,2,2,"[""kc_interaction__polarity_negative__and__voice_passive"", ""kc_cell__tense_present__aspect_none__voice_passive__polar..."
3,activation_class_004,kc_interaction__aspect_progressive__and__voice_passive,2,2,"[""kc_interaction__aspect_progressive__and__voice_passive"", ""kc_cell__tense_present__aspect_progressive__voice_passiv..."
4,activation_class_006,kc_feature__voice__passive,2,5,"[""kc_feature__voice__passive"", ""kc_operation__passive_dependency""]"
5,activation_class_008,kc_interaction__aspect_perfect_progressive__and__polarity_negative,2,2,"[""kc_interaction__aspect_perfect_progressive__and__polarity_negative"", ""kc_cell__tense_past__aspect_perfect_progress..."
6,activation_class_010,kc_interaction__aspect_perfect_progressive__and__tense_present,2,2,"[""kc_interaction__aspect_perfect_progressive__and__tense_present"", ""kc_cell__tense_present__aspect_perfect_progressi..."
7,activation_class_022,kc_interaction__aspect_progressive__and__polarity_negative,2,2,"[""kc_interaction__aspect_progressive__and__polarity_negative"", ""kc_cell__tense_present__aspect_progressive__voice_ac..."
8,activation_class_025,kc_feature__aspect__progressive,2,6,"[""kc_feature__aspect__progressive"", ""kc_interaction__aspect_progressive__and__tense_present""]"
9,activation_class_027,kc_interaction__clause_imperative__and__polarity_negative,2,2,"[""kc_interaction__clause_imperative__and__polarity_negative"", ""kc_cell__tense_na__aspect_none__voice_active__polarit..."


## 8. Learner evidence: development acquisition and frozen probes

The mixed synthetic world combines reusable feature, interaction, and weaker exact-cell components. Acquisition contains development items only; every grammar regime is measured in the frozen probe phase. The private oracle-debug artifact is intentionally not loaded.

In [11]:
world = read_yaml('simulation/materialized_world.yaml')
world_overview = pd.DataFrame([{
    'world_id': world['world_id'],
    'claim': world['claim'],
    'seed': world['seed'],
    'learners': world['learners'],
    'passes': world['passes'],
    'item_order': world['item_order'],
    'train_fraction': world['temporal_split']['train_fraction'],
    'validation_fraction': world['temporal_split']['validation_fraction'],
    'test_fraction': world['temporal_split']['test_fraction'],
}])
world_components = pd.json_normalize(world['latent_structure']['components'], sep='.').map(compact)
events = read_jsonl('simulation/events.jsonl.gz')
event_summary = events.groupby(['protocol_phase', 'dataset_split', 'grammar_split']).agg(
    events=('event_id', 'size'),
    learners=('learner_id', 'nunique'),
    items=('item_id', 'nunique'),
    correctness=('correct', 'mean'),
    updates_history_rate=('updates_history', 'mean'),
    updates_mastery_rate=('updates_mastery', 'mean'),
).reset_index()
per_learner = events.groupby('learner_id').agg(
    events=('event_id', 'size'),
    first_sequence_index=('sequence_index', 'min'),
    last_sequence_index=('sequence_index', 'max'),
    correctness=('correct', 'mean'),
).reset_index()
learner_distribution = per_learner[['events', 'first_sequence_index', 'last_sequence_index', 'correctness']].describe().T.reset_index().rename(columns={'index': 'measure'})
event_contract = pd.DataFrame([{
    'events': len(events),
    'learners': events['learner_id'].nunique(),
    'items': events['item_id'].nunique(),
    'acquisition_events': int(events['protocol_phase'].eq('acquisition').sum()),
    'probe_events': int(events['protocol_phase'].eq('probe').sum()),
    'holdout_acquisition_events': int(((events['protocol_phase'] == 'acquisition') & (events['grammar_split'] != 'development')).sum()),
    'probe_events_updating_mastery': int(((events['protocol_phase'] == 'probe') & events['updates_mastery']).sum()),
}])
show_table('Materialized latent-world declaration', world_overview)
show_table('Latent-world structural components', world_components)
show_table('Frozen-probe event contract', event_contract)
show_table('Learner events by protocol phase, temporal split, and grammar regime', event_summary)
show_table('Per-learner event distribution', learner_distribution)
show_table('Chronological learner-event sample', events.head(EVENT_SAMPLE_ROWS), limit=EVENT_SAMPLE_ROWS)

**Materialized latent-world declaration** — 1 row(s)

,world_id,claim,seed,learners,passes,item_order,train_fraction,validation_fraction,test_fraction
0,phase4_mixed_v1,"Responses combine reusable feature knowledge, selected interaction-specific knowledge, and a weaker exact-cell compo...",20260827,1000,6,counterbalanced_rotate_by_learner_and_pass,0.6,0.2,0.2


**Latent-world structural components** — 3 row(s)

,kind,initial_mastery_beta,learning_rate,weight,background_values.tense,background_values.aspect,background_values.voice,background_values.polarity,background_values.clause,background_values.modal,interactions,scope
0,feature_values,"[3.0, 3.0]",0.070,1.00,"[""NA""]","[""none""]","[""active""]","[""positive""]","[""declarative""]","[""none""]",NaN,NaN
1,declared_interactions,"[1.2, 6.8]",0.025,1.50,NaN,NaN,NaN,NaN,NaN,NaN,"[{""activation"": {""aspect"": ""perfect"", ""polarity"": ""negative""}, ""id"": ""perfect_negative""}, {""activation"": {""polarity""...",NaN
2,exact_cells,"[2.0, 4.0]",0.035,0.75,NaN,NaN,NaN,NaN,NaN,NaN,NaN,all_cells


**Frozen-probe event contract** — 1 row(s)

...

**Per-learner event distribution** — 4 row(s)

,measure,count,mean,std,min,25%,50%,75%,max
0,events,1000.0,204.000000,0.000000,204.000000,204.000000,204.000000,204.000000,204.000000
1,first_sequence_index,1000.0,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000
2,last_sequence_index,1000.0,204.000000,0.000000,204.000000,204.000000,204.000000,204.000000,204.000000
3,correctness,1000.0,0.600877,0.036142,0.485294,0.578431,0.602941,0.627451,0.715686


**Chronological learner-event sample** — 20 row(s)

,correct,dataset_split,event_id,grammar_split,item_difficulty,item_id,learner_id,protocol_phase,sequence_index,updates_history,updates_mastery
0,0,train,event_0000001,development,0.15,candidate_cell_001_01,learner_001,acquisition,1,True,True
1,0,train,event_0000002,development,0.15,candidate_cell_001_02,learner_001,acquisition,2,True,True
2,0,train,event_0000003,development,0.15,candidate_cell_002_01,learner_001,acquisition,3,True,True
3,1,train,event_0000004,development,0.15,candidate_cell_002_03,learner_001,acquisition,4,True,True
4,1,train,event_0000005,development,0.00,candidate_cell_003_01,learner_001,acquisition,5,True,True
5,1,train,event_0000006,development,0.00,candidate_cell_003_03,learner_001,acquisition,6,True,True
6,1,train,event_0000007,development,0.00,candidate_cell_004_01,learner_001,acquisition,7,True,True
7,0,train,event_0000008,development,0.00,candidate_cell_004_02,learner_001,acquisition,8,True,True
8,1,train,event_0000009,development,0.15,candidate_cell_005_03,learner_001,acquisition,9,True,True
9,0,train,event_0000010,development,0.15,candidate_cell_006_01,learner_001,acquisition,10,True,True


,correct,dataset_split,event_id,grammar_split,item_difficulty,item_id,learner_id,protocol_phase,sequence_index,updates_history,updates_mastery
0,0,train,event_0000001,development,0.15,candidate_cell_001_01,learner_001,acquisition,1,True,True
1,0,train,event_0000002,development,0.15,candidate_cell_001_02,learner_001,acquisition,2,True,True
2,0,train,event_0000003,development,0.15,candidate_cell_002_01,learner_001,acquisition,3,True,True
3,1,train,event_0000004,development,0.15,candidate_cell_002_03,learner_001,acquisition,4,True,True
4,1,train,event_0000005,development,0.00,candidate_cell_003_01,learner_001,acquisition,5,True,True
5,1,train,event_0000006,development,0.00,candidate_cell_003_03,learner_001,acquisition,6,True,True
6,1,train,event_0000007,development,0.00,candidate_cell_004_01,learner_001,acquisition,7,True,True
7,0,train,event_0000008,development,0.00,candidate_cell_004_02,learner_001,acquisition,8,True,True
8,1,train,event_0000009,development,0.15,candidate_cell_005_03,learner_001,acquisition,9,True,True
9,0,train,event_0000010,development,0.15,candidate_cell_006_01,learner_001,acquisition,10,True,True


## 9. Development-only automated KC selection

The selector starts from reusable feature-value KCs, greedily tests structurally eligible additions under validation log loss plus a per-KC complexity penalty, and backward-prunes. Holdout grammar and reserved test outcomes remain outside selection.

In [12]:
selection_trace = read_json('kc/selection_trace.json')
selection_design_table = pd.DataFrame([{
    'selection_id': selection_trace['selection_id'],
    'selector_model': selection_trace['selector_model']['model'],
    'metric': selection_trace['objective']['metric'],
    'complexity_measure': selection_trace['objective']['complexity'],
    'complexity_penalty_per_KC': selection_trace['objective']['complexity_penalty'],
    'split_mode': selection_trace['split']['mode'],
    'train_events': selection_trace['split']['train_events'],
    'validation_events': selection_trace['split']['validation_events'],
    'held_out_grammar_read': selection_trace['held_out_grammar_read'],
    'reserved_or_holdout_outcomes_read': selection_trace['reserved_or_holdout_outcomes_read'],
    'selected_KCs': len(selection_trace['selected_candidate_ids']),
    'final_validation_log_loss': selection_trace['final_validation_score']['log_loss'],
    'final_objective': selection_trace['final_validation_score']['objective'],
}])
selection_steps = pd.DataFrame([
    {
        'step': row['step'],
        'action': row['action'],
        'selected': row.get('selected'),
        'best_rejected': row.get('best_rejected', {}).get('candidate_id'),
        'validation_n': row.get('score', row.get('best_rejected', {}).get('score', {})).get('n'),
        'log_loss': row.get('score', row.get('best_rejected', {}).get('score', {})).get('log_loss'),
        'KC_count': row.get('score', row.get('best_rejected', {}).get('score', {})).get('kc_count'),
        'complexity_penalty': row.get('score', row.get('best_rejected', {}).get('score', {})).get('complexity_penalty'),
        'objective': row.get('score', row.get('best_rejected', {}).get('score', {})).get('objective'),
        'objective_improvement': row.get('objective_improvement', row.get('best_rejected', {}).get('objective_improvement')),
    }
    for row in selection_trace['trace']
])
candidate_score_rows = []
for row in selection_trace['trace']:
    for scored in row.get('candidate_scores', []):
        candidate_score_rows.append({
            'step': row['step'],
            'candidate_id': scored['candidate_id'],
            'log_loss': scored['score']['log_loss'],
            'KC_count': scored['score']['kc_count'],
            'complexity_penalty': scored['score']['complexity_penalty'],
            'objective': scored['score']['objective'],
            'objective_improvement': scored['objective_improvement'],
        })
candidate_scores = pd.DataFrame(candidate_score_rows)
selected_kcs = kc_candidates_raw.loc[kc_candidates_raw['id'].isin(selection_trace['selected_candidate_ids']), [
    'id', 'family', 'definition', 'cell_support', 'item_support', 'equivalence_class_id',
]].copy()
selected_kcs['selection_order'] = selected_kcs['id'].map({candidate_id: index + 1 for index, candidate_id in enumerate(selection_trace['selected_candidate_ids'])})
selected_kcs = selected_kcs.sort_values('selection_order')
show_table('KC selection design, split, and leakage checks', selection_design_table)
show_table('Automated selection trajectory', selection_steps)
show_table('Candidate scores considered during forward search', candidate_scores.sort_values(['step', 'objective']))
show_table('Frozen automatically selected KC inventory', selected_kcs)

**KC selection design, split, and leakage checks** — 1 row(s)

,selection_id,selector_model,metric,complexity_measure,complexity_penalty_per_KC,split_mode,train_events,validation_events,held_out_grammar_read,reserved_or_holdout_outcomes_read,selected_KCs,final_validation_log_loss,final_objective
0,forward_predictive_parsimony_v1,observable_logistic_kt,log_loss,kc_count,0.0005,chronological,128000,32000,False,False,10,0.646354,0.651354


**Automated selection trajectory** — 3 row(s)

,step,action,selected,best_rejected,validation_n,log_loss,KC_count,complexity_penalty,objective,objective_improvement
0,0,initial_factorized,None,None,32000,0.647160,9,0.0045,0.651660,NaN
1,1,forward_add,kc_interaction__aspect_perfect__and__polarity_negative,None,32000,0.646354,10,0.0050,0.651354,0.000307
2,2,forward_stop,None,kc_interaction__polarity_negative__and__tense_present,32000,0.645991,11,0.0055,0.651491,-0.000137


**Candidate scores considered during forward search** — 19 row(s)

,step,candidate_id,log_loss,KC_count,complexity_penalty,objective,objective_improvement
0,1,kc_interaction__aspect_perfect__and__polarity_negative,0.646354,10,0.0050,0.651354,0.000307
1,1,kc_interaction__polarity_negative__and__tense_present,0.646828,10,0.0050,0.651828,-0.000168
2,1,kc_operation__finite_tense_form,0.646966,10,0.0050,0.651966,-0.000306
3,1,kc_interaction__tense_present__and__voice_passive,0.646997,10,0.0050,0.651997,-0.000337
4,1,kc_operation__progressive_dependency,0.647094,10,0.0050,0.652094,-0.000433
5,1,kc_interaction__aspect_perfect_progressive__and__tense_past,0.647113,10,0.0050,0.652113,-0.000452
6,1,kc_operation__perfect_dependency,0.647170,10,0.0050,0.652170,-0.000510
7,1,kc_interaction__aspect_perfect__and__tense_present,0.647197,10,0.0050,0.652197,-0.000537
8,1,kc_interaction__aspect_perfect__and__tense_past,0.647208,10,0.0050,0.652208,-0.000548
9,1,kc_interaction__polarity_negative__and__tense_past,0.647370,10,0.0050,0.652370,-0.000710


**Frozen automatically selected KC inventory** — 10 row(s)

,id,family,definition,cell_support,item_support,equivalence_class_id,selection_order
0,kc_feature__aspect__perfect,feature_value,Represent canonical aspect=perfect.,5,8,activation_class_020,1
1,kc_feature__aspect__perfect_progressive,feature_value,Represent canonical aspect=perfect_progressive.,3,5,activation_class_011,2
2,kc_feature__aspect__progressive,feature_value,Represent canonical aspect=progressive.,3,6,activation_class_025,3
3,kc_feature__clause__imperative,feature_value,Represent canonical clause=imperative.,2,3,activation_class_030,4
4,kc_feature__clause__polar_question,feature_value,Represent canonical clause=polar_question.,2,4,activation_class_037,5
5,kc_feature__polarity__negative,feature_value,Represent canonical polarity=negative.,6,12,activation_class_028,6
6,kc_feature__tense__past,feature_value,Represent canonical tense=past.,7,11,activation_class_034,7
7,kc_feature__tense__present,feature_value,Represent canonical tense=present.,9,18,activation_class_036,8
8,kc_feature__voice__passive,feature_value,Represent canonical voice=passive.,3,5,activation_class_006,9
9,kc_interaction__aspect_perfect__and__polarity_negative,interaction,Represent the supported interaction aspect=perfect and polarity=negative.,2,4,activation_class_018,10


,id,family,definition,cell_support,item_support,equivalence_class_id,selection_order
0,kc_feature__aspect__perfect,feature_value,Represent canonical aspect=perfect.,5,8,activation_class_020,1
1,kc_feature__aspect__perfect_progressive,feature_value,Represent canonical aspect=perfect_progressive.,3,5,activation_class_011,2
2,kc_feature__aspect__progressive,feature_value,Represent canonical aspect=progressive.,3,6,activation_class_025,3
3,kc_feature__clause__imperative,feature_value,Represent canonical clause=imperative.,2,3,activation_class_030,4
4,kc_feature__clause__polar_question,feature_value,Represent canonical clause=polar_question.,2,4,activation_class_037,5
5,kc_feature__polarity__negative,feature_value,Represent canonical polarity=negative.,6,12,activation_class_028,6
6,kc_feature__tense__past,feature_value,Represent canonical tense=past.,7,11,activation_class_034,7
7,kc_feature__tense__present,feature_value,Represent canonical tense=present.,9,18,activation_class_036,8
8,kc_feature__voice__passive,feature_value,Represent canonical voice=passive.,3,5,activation_class_006,9
9,kc_interaction__aspect_perfect__and__polarity_negative,interaction,Represent the supported interaction aspect=perfect and polarity=negative.,2,4,activation_class_018,10


## 10. Frozen KC policies, item–KC projections, and Q-matrices

The automated policy and comparison policies are projected over the identical fixed item bank. The Q-matrix tables make item coverage, edge count, density, and KCs per item directly inspectable.

In [13]:
policy_summary_rows = []
policy_kc_rows = []
for policy_path in sorted((DATA_DIR / 'kc/policies').glob('*.yaml')):
    policy_name = policy_path.stem
    policy = read_yaml(f'kc/policies/{policy_path.name}')
    q_matrix = pd.read_csv(DATA_DIR / f'kc/q_matrices/{policy_name}.csv')
    kc_columns = [column for column in q_matrix if column != 'item_id']
    edge_count = int(q_matrix[kc_columns].to_numpy().sum()) if kc_columns else 0
    policy_summary_rows.append({
        'representation': policy_name,
        'policy_id': policy['policy_id'],
        'items': len(q_matrix),
        'KCs': len(kc_columns),
        'Q_edges': edge_count,
        'Q_density': edge_count / (len(q_matrix) * len(kc_columns)) if kc_columns else 0.0,
        'KCs_per_item': edge_count / len(q_matrix),
        'uncovered_items': int(q_matrix[kc_columns].sum(axis=1).eq(0).sum()) if kc_columns else len(q_matrix),
    })
    declared_kcs = policy.get('kcs', [
        {
            'id': kc_id,
            'definition': policy['description'],
            'activation': {'generated_by': policy.get('kc_id_pattern', policy.get('kind', 'declared_pattern'))},
        }
        for kc_id in kc_columns
    ])
    for kc in declared_kcs:
        policy_kc_rows.append({
            'representation': policy_name,
            'KC_id': kc['id'],
            'definition': kc['definition'],
            'activation': compact(kc['activation']),
        })
policy_summary = pd.DataFrame(policy_summary_rows).sort_values('KCs')
policy_kcs = pd.DataFrame(policy_kc_rows)
automated_projection = read_jsonl('kc/projections/automated.jsonl')
automated_projection['kc_ids'] = automated_projection['kc_ids'].map(compact)
automated_q_matrix = pd.read_csv(DATA_DIR / 'kc/q_matrices/automated.csv')
show_table('Frozen policy granularity and Q-matrix diagnostics', policy_summary)
show_table('KCs declared by each frozen policy', policy_kcs)
show_table('Automated item→KC projection', automated_projection)
show_table('Automated-policy Q-matrix', automated_q_matrix)

**Frozen policy granularity and Q-matrix diagnostics** — 4 row(s)

,representation,policy_id,items,KCs,Q_edges,Q_density,KCs_per_item,uncovered_items
0,factorized,medium_v1_factorized,44,9,92,0.232323,2.090909,2
1,automated,selected__forward_predictive_parsimony_v1,44,10,96,0.218182,2.181818,2
2,supported_interactions,medium_v1_factorized_plus_all_supported_interactions,44,16,127,0.180398,2.886364,2
3,oracle_all_cell,medium_v1_oracle_exact_all_cell,44,24,44,0.041667,1.000000,0


**KCs declared by each frozen policy** — 59 row(s)

,representation,KC_id,definition,activation
0,automated,kc_feature__aspect__perfect,Represent canonical aspect=perfect.,"{""cell"": {""aspect"": ""perfect""}}"
1,automated,kc_feature__aspect__perfect_progressive,Represent canonical aspect=perfect_progressive.,"{""cell"": {""aspect"": ""perfect_progressive""}}"
2,automated,kc_feature__aspect__progressive,Represent canonical aspect=progressive.,"{""cell"": {""aspect"": ""progressive""}}"
3,automated,kc_feature__clause__imperative,Represent canonical clause=imperative.,"{""cell"": {""clause"": ""imperative""}}"
4,automated,kc_feature__clause__polar_question,Represent canonical clause=polar_question.,"{""cell"": {""clause"": ""polar_question""}}"
5,automated,kc_feature__polarity__negative,Represent canonical polarity=negative.,"{""cell"": {""polarity"": ""negative""}}"
6,automated,kc_feature__tense__past,Represent canonical tense=past.,"{""cell"": {""tense"": ""past""}}"
7,automated,kc_feature__tense__present,Represent canonical tense=present.,"{""cell"": {""tense"": ""present""}}"
8,automated,kc_feature__voice__passive,Represent canonical voice=passive.,"{""cell"": {""voice"": ""passive""}}"
9,automated,kc_interaction__aspect_perfect__and__polarity_negative,Represent the supported interaction aspect=perfect and polarity=negative.,"{""cell"": {""aspect"": ""perfect"", ""polarity"": ""negative""}}"


**Automated item→KC projection** — 44 row(s)

,item_id,kc_ids
0,candidate_cell_001_01,"[""kc_feature__clause__polar_question"", ""kc_feature__tense__present""]"
1,candidate_cell_001_02,"[""kc_feature__clause__polar_question"", ""kc_feature__tense__present""]"
2,candidate_cell_002_01,"[""kc_feature__clause__polar_question"", ""kc_feature__tense__past""]"
3,candidate_cell_002_03,"[""kc_feature__clause__polar_question"", ""kc_feature__tense__past""]"
4,candidate_cell_003_01,"[""kc_feature__tense__present""]"
5,candidate_cell_003_03,"[""kc_feature__tense__present""]"
6,candidate_cell_004_01,"[""kc_feature__tense__past""]"
7,candidate_cell_004_02,"[""kc_feature__tense__past""]"
8,candidate_cell_005_03,"[""kc_feature__clause__imperative""]"
9,candidate_cell_006_01,"[""kc_feature__clause__imperative"", ""kc_feature__polarity__negative""]"


**Automated-policy Q-matrix** — 44 row(s)

,item_id,kc_feature__aspect__perfect,kc_feature__aspect__perfect_progressive,kc_feature__aspect__progressive,kc_feature__clause__imperative,kc_feature__clause__polar_question,kc_feature__polarity__negative,kc_feature__tense__past,kc_feature__tense__present,kc_feature__voice__passive,kc_interaction__aspect_perfect__and__polarity_negative
0,candidate_cell_001_01,0,0,0,0,1,0,0,1,0,0
1,candidate_cell_001_02,0,0,0,0,1,0,0,1,0,0
2,candidate_cell_002_01,0,0,0,0,1,0,1,0,0,0
3,candidate_cell_002_03,0,0,0,0,1,0,1,0,0,0
4,candidate_cell_003_01,0,0,0,0,0,0,0,1,0,0
5,candidate_cell_003_03,0,0,0,0,0,0,0,1,0,0
6,candidate_cell_004_01,0,0,0,0,0,0,1,0,0,0
7,candidate_cell_004_02,0,0,0,0,0,0,1,0,0,0
8,candidate_cell_005_03,0,0,0,1,0,0,0,0,0,0
9,candidate_cell_006_01,0,0,0,1,0,1,0,0,0,0


,item_id,kc_feature__aspect__perfect,kc_feature__aspect__perfect_progressive,kc_feature__aspect__progressive,kc_feature__clause__imperative,kc_feature__clause__polar_question,kc_feature__polarity__negative,kc_feature__tense__past,kc_feature__tense__present,kc_feature__voice__passive,kc_interaction__aspect_perfect__and__polarity_negative
0,candidate_cell_001_01,0,0,0,0,1,0,0,1,0,0
1,candidate_cell_001_02,0,0,0,0,1,0,0,1,0,0
2,candidate_cell_002_01,0,0,0,0,1,0,1,0,0,0
3,candidate_cell_002_03,0,0,0,0,1,0,1,0,0,0
4,candidate_cell_003_01,0,0,0,0,0,0,0,1,0,0
5,candidate_cell_003_03,0,0,0,0,0,0,0,1,0,0
6,candidate_cell_004_01,0,0,0,0,0,0,1,0,0,0
7,candidate_cell_004_02,0,0,0,0,0,0,1,0,0,0
8,candidate_cell_005_03,0,0,0,1,0,0,0,0,0,0
9,candidate_cell_006_01,0,0,0,1,0,1,0,0,0,0


## 11. Online knowledge-tracing predictions

Each representation is evaluated with empirical, BKT, and observable logistic KT predictions over the same 204,000 learner events. To keep the notebook responsive, only the automated prediction file is loaded fully; the other retained files are inventoried and their aggregate metrics are read in the evaluation stage.

In [14]:
prediction_artifacts = []
for prediction_path in sorted((DATA_DIR / 'kt').glob('*/predictions.jsonl.gz')):
    representation = prediction_path.parent.name
    result_path = DATA_DIR / f'evaluation/{representation}/results.json'
    expected_rows = read_json(f'evaluation/{representation}/results.json')['input_counts']['predictions'] if result_path.is_file() else pd.NA
    prediction_artifacts.append({
        'representation': representation,
        'relative_path': str(prediction_path.relative_to(DATA_DIR)),
        'compressed_size_bytes': prediction_path.stat().st_size,
        'prediction_rows': expected_rows,
        'techniques': 3,
    })
prediction_artifact_table = pd.DataFrame(prediction_artifacts)
automated_predictions = read_jsonl('kt/automated/predictions.jsonl.gz')
prediction_summary = automated_predictions.groupby('technique').agg(
    predictions=('event_id', 'size'),
    events=('event_id', 'nunique'),
    mean_probability=('probability', 'mean'),
    minimum_probability=('probability', 'min'),
    maximum_probability=('probability', 'max'),
    median_history_events=('history_events', 'median'),
).reset_index()
prediction_sample = automated_predictions.groupby('technique', group_keys=False).head(PREDICTION_SAMPLE_ROWS_PER_TECHNIQUE)
prediction_sample = prediction_sample.merge(
    events[['event_id', 'learner_id', 'item_id', 'correct', 'dataset_split', 'grammar_split', 'protocol_phase']],
    on='event_id', how='left',
)
show_table('Retained KT prediction artifacts', prediction_artifact_table)
show_table('Automated-policy prediction summary by KT technique', prediction_summary)
show_table('Automated-policy prediction sample joined to learner events', prediction_sample, limit=len(prediction_sample))

**Retained KT prediction artifacts** — 4 row(s)

,representation,relative_path,compressed_size_bytes,prediction_rows,techniques
0,automated,kt/automated/predictions.jsonl.gz,7605964,612000,3
1,factorized,kt/factorized/predictions.jsonl.gz,7586675,612000,3
2,oracle_all_cell,kt/oracle_all_cell/predictions.jsonl.gz,5711641,612000,3
3,supported_interactions,kt/supported_interactions/predictions.jsonl.gz,7703013,612000,3


**Automated-policy prediction summary by KT technique** — 3 row(s)

,technique,predictions,events,mean_probability,minimum_probability,maximum_probability,median_history_events
0,bkt,204000,204000,0.777030,0.278400,0.900000,101.5
1,empirical,204000,204000,0.539104,0.052632,0.916667,101.5
2,logistic,204000,204000,0.599674,0.290747,0.724745,101.5


**Automated-policy prediction sample joined to learner events** — 24 row(s)

,event_id,history_events,probability,technique,learner_id,item_id,correct,dataset_split,grammar_split,protocol_phase
0,event_0000001,0,0.500000,empirical,learner_001,candidate_cell_001_01,0,train,development,acquisition
1,event_0000002,1,0.333333,empirical,learner_001,candidate_cell_001_02,0,train,development,acquisition
2,event_0000003,2,0.375000,empirical,learner_001,candidate_cell_002_01,0,train,development,acquisition
3,event_0000004,3,0.266667,empirical,learner_001,candidate_cell_002_03,1,train,development,acquisition
4,event_0000005,4,0.250000,empirical,learner_001,candidate_cell_003_01,1,train,development,acquisition
5,event_0000006,5,0.400000,empirical,learner_001,candidate_cell_003_03,1,train,development,acquisition
6,event_0000007,6,0.500000,empirical,learner_001,candidate_cell_004_01,1,train,development,acquisition
7,event_0000008,7,0.600000,empirical,learner_001,candidate_cell_004_02,0,train,development,acquisition
8,event_0000001,0,0.432000,bkt,learner_001,candidate_cell_001_01,0,train,development,acquisition
9,event_0000002,1,0.305442,bkt,learner_001,candidate_cell_001_02,0,train,development,acquisition


,event_id,history_events,probability,technique,learner_id,item_id,correct,dataset_split,grammar_split,protocol_phase
0,event_0000001,0,0.500000,empirical,learner_001,candidate_cell_001_01,0,train,development,acquisition
1,event_0000002,1,0.333333,empirical,learner_001,candidate_cell_001_02,0,train,development,acquisition
2,event_0000003,2,0.375000,empirical,learner_001,candidate_cell_002_01,0,train,development,acquisition
3,event_0000004,3,0.266667,empirical,learner_001,candidate_cell_002_03,1,train,development,acquisition
4,event_0000005,4,0.250000,empirical,learner_001,candidate_cell_003_01,1,train,development,acquisition
5,event_0000006,5,0.400000,empirical,learner_001,candidate_cell_003_03,1,train,development,acquisition
6,event_0000007,6,0.500000,empirical,learner_001,candidate_cell_004_01,1,train,development,acquisition
7,event_0000008,7,0.600000,empirical,learner_001,candidate_cell_004_02,0,train,development,acquisition
8,event_0000001,0,0.432000,bkt,learner_001,candidate_cell_001_01,0,train,development,acquisition
9,event_0000002,1,0.305442,bkt,learner_001,candidate_cell_001_02,0,train,development,acquisition


## 12. Predictive evaluation and learner-paired uncertainty

The long metrics table holds representation fixed within each row and reports empirical, BKT, and logistic results overall and by grammar regime. The paired table fixes logistic KT and bootstraps learners; negative candidate-minus-reference deltas favor the candidate.

In [15]:
representation_rows = []
metric_rows = []
for result_path in sorted((DATA_DIR / 'evaluation').glob('*/results.json')):
    representation_name = result_path.parent.name
    result = read_json(f'evaluation/{representation_name}/results.json')
    representation_rows.append({'representation': representation_name, **{
        key: compact(value) for key, value in result['representation'].items() if key != 'kc_support'
    }})
    for technique, technique_metrics in result['kt'].items():
        overall = {key: technique_metrics[key] for key in ('n', 'log_loss', 'brier_score', 'auc', 'ece', 'accuracy')}
        metric_rows.append({'representation': representation_name, 'technique': technique, 'grammar_regime': 'all_test', **overall})
        for grammar_regime, regime_metrics in technique_metrics['grammar_split_metrics'].items():
            metric_rows.append({'representation': representation_name, 'technique': technique, 'grammar_regime': grammar_regime, **regime_metrics})
representation_results = pd.DataFrame(representation_rows).sort_values('kcs')
kt_metrics = pd.DataFrame(metric_rows).sort_values(['technique', 'grammar_regime', 'log_loss'])
paired = read_json('evaluation/paired_logistic.json')
paired_rows = pd.DataFrame([
    {
        'grammar_regime': row['grammar_regime'],
        'reference': row['reference'],
        'candidate': row['candidate'],
        'learners': row['n_learners'],
        'events': row['n_events'],
        'delta_log_loss': row['delta_log_loss']['point_estimate'],
        'log_loss_CI_low': row['delta_log_loss']['interval_95'][0],
        'log_loss_CI_high': row['delta_log_loss']['interval_95'][1],
        'delta_Brier': row['delta_brier_score']['point_estimate'],
        'Brier_CI_low': row['delta_brier_score']['interval_95'][0],
        'Brier_CI_high': row['delta_brier_score']['interval_95'][1],
        'interval_excludes_zero': not (row['delta_log_loss']['interval_95'][0] <= 0 <= row['delta_log_loss']['interval_95'][1]),
    }
    for row in paired['comparisons'] if row['available']
])
show_table('Representation granularity and coverage results', representation_results)
show_table('KT metrics by representation, technique, and grammar regime', kt_metrics)
show_table('Fixed-logistic learner-cluster paired comparisons', paired_rows.sort_values(['grammar_regime', 'reference', 'candidate']))

**Representation granularity and coverage results** — 4 row(s)

,representation,policy_id,items,kcs,item_coverage,event_coverage,q_matrix_density,kcs_per_item,redundant_kcs,compositional_coverage
0,factorized,medium_v1_factorized,44,9,0.954545,0.990196,0.232323,2.090909,[],1.0
1,automated,selected__forward_predictive_parsimony_v1,44,10,0.954545,0.990196,0.218182,2.181818,[],1.0
2,supported_interactions,medium_v1_factorized_plus_all_supported_interactions,44,16,0.954545,0.990196,0.180398,2.886364,[],1.0
3,oracle_all_cell,medium_v1_oracle_exact_all_cell,44,24,1.000000,1.000000,0.041667,1.000000,[],0.0


**KT metrics by representation, technique, and grammar regime** — 48 row(s)

,representation,technique,grammar_regime,n,log_loss,brier_score,auc,ece,accuracy
0,oracle_all_cell,bkt,all_test,44000,0.772235,0.270253,0.529215,0.193087,0.562977
1,supported_interactions,bkt,all_test,44000,0.817456,0.275769,0.542285,0.213057,0.641795
2,automated,bkt,all_test,44000,0.826163,0.277927,0.543480,0.219361,0.642295
3,factorized,bkt,all_test,44000,0.827526,0.278252,0.543681,0.220417,0.642295
4,oracle_all_cell,bkt,compositional_holdout,10000,0.740744,0.273637,0.500000,0.207800,0.360200
5,supported_interactions,bkt,compositional_holdout,10000,0.843014,0.284479,0.534308,0.228685,0.638800
6,automated,bkt,compositional_holdout,10000,0.855793,0.287766,0.537843,0.235326,0.639600
7,factorized,bkt,compositional_holdout,10000,0.855793,0.287766,0.537843,0.235326,0.639600
8,oracle_all_cell,bkt,development,32000,0.788000,0.270950,0.516063,0.202027,0.624563
9,supported_interactions,bkt,development,32000,0.813955,0.273055,0.518088,0.213981,0.657125


**Fixed-logistic learner-cluster paired comparisons** — 12 row(s)

,grammar_regime,reference,candidate,learners,events,delta_log_loss,log_loss_CI_low,log_loss_CI_high,delta_Brier,Brier_CI_low,Brier_CI_high,interval_excludes_zero
0,all_test,factorized,automated,1000,44000,-0.000375,-0.000631,-0.000109,-0.000166,-0.000281,-0.000046,True
1,all_test,factorized,oracle_all_cell,1000,44000,0.013777,0.012288,0.015234,0.006789,0.006070,0.007491,True
2,all_test,factorized,supported_interactions,1000,44000,-0.000397,-0.000782,-0.000026,-0.000180,-0.000356,-0.000008,True
3,compositional_holdout,factorized,automated,1000,10000,-0.000234,-0.000836,0.000375,-0.000091,-0.000372,0.000192,False
4,compositional_holdout,factorized,oracle_all_cell,1000,10000,0.059615,0.053390,0.065675,0.029375,0.026351,0.032305,True
5,compositional_holdout,factorized,supported_interactions,1000,10000,-0.001168,-0.002042,-0.000246,-0.000532,-0.000935,-0.000109,True
6,development,factorized,automated,1000,32000,-0.000450,-0.000758,-0.000141,-0.000203,-0.000339,-0.000065,True
7,development,factorized,oracle_all_cell,1000,32000,-0.000234,-0.000753,0.000271,-0.000115,-0.000348,0.000112,False
8,development,factorized,supported_interactions,1000,32000,-0.000176,-0.000605,0.000252,-0.000079,-0.000276,0.000117,False
9,novel_feature_holdout,factorized,automated,1000,2000,0.000119,-0.000099,0.000352,0.000057,-0.000047,0.000169,False


,grammar_regime,reference,candidate,learners,events,delta_log_loss,log_loss_CI_low,log_loss_CI_high,delta_Brier,Brier_CI_low,Brier_CI_high,interval_excludes_zero
0,all_test,factorized,automated,1000,44000,-0.000375,-0.000631,-0.000109,-0.000166,-0.000281,-0.000046,True
1,all_test,factorized,oracle_all_cell,1000,44000,0.013777,0.012288,0.015234,0.006789,0.006070,0.007491,True
2,all_test,factorized,supported_interactions,1000,44000,-0.000397,-0.000782,-0.000026,-0.000180,-0.000356,-0.000008,True
3,compositional_holdout,factorized,automated,1000,10000,-0.000234,-0.000836,0.000375,-0.000091,-0.000372,0.000192,False
4,compositional_holdout,factorized,oracle_all_cell,1000,10000,0.059615,0.053390,0.065675,0.029375,0.026351,0.032305,True
5,compositional_holdout,factorized,supported_interactions,1000,10000,-0.001168,-0.002042,-0.000246,-0.000532,-0.000935,-0.000109,True
6,development,factorized,automated,1000,32000,-0.000450,-0.000758,-0.000141,-0.000203,-0.000339,-0.000065,True
7,development,factorized,oracle_all_cell,1000,32000,-0.000234,-0.000753,0.000271,-0.000115,-0.000348,0.000112,False
8,development,factorized,supported_interactions,1000,32000,-0.000176,-0.000605,0.000252,-0.000079,-0.000276,0.000117,False
9,novel_feature_holdout,factorized,automated,1000,2000,0.000119,-0.000099,0.000352,0.000057,-0.000047,0.000169,False


## 13. KC-selection support and seed stability

Nested learner prefixes test evidence requirements; five independent 1,000-learner streams test full-support seed stability. This is selection stability over the fixed bank and fixed candidate inventory, not empirical learner replication.

In [16]:
stability = read_json('kc/selection_stability.json')
stability_conditions = pd.DataFrame([
    {
        'condition_id': row['condition_id'],
        'seed': row['seed'],
        'learners': row['learners'],
        'stage': row['stage'],
        'selection_events': row['selection_events'],
        'validation_events': row['selection_validation_events'],
        'KC_count': row['kc_count'],
        'addition_count': row['addition_count'],
        'selected_additions': compact(row['selected_addition_ids']),
        'exact_reference_match': row['exact_selected_inventory_match_reference'],
        'validation_log_loss': row['final_validation_score']['log_loss'],
        'objective': row['final_validation_score']['objective'],
    }
    for row in stability['selections']
])
full_support_frequencies = pd.DataFrame(stability['frequencies']['five_seed_full_support']['selected'])
if not full_support_frequencies.empty:
    full_support_frequencies['condition_ids'] = full_support_frequencies['condition_ids'].map(compact)
jaccard_summary_rows = []
for scope, summary in stability['jaccard'].items():
    for inventory_type in ('all_selected', 'additions'):
        jaccard_summary_rows.append({'scope': scope, 'inventory': inventory_type, **summary[inventory_type]})
jaccard_summary = pd.DataFrame(jaccard_summary_rows)
stability_contract = pd.DataFrame([{
    **stability['exact_inventory_matches'],
    **stability['boundary_checks'],
}])
show_table('Selection stability conditions', stability_conditions)
show_table('Full-support five-seed KC selection frequency', full_support_frequencies)
show_table('Inventory Jaccard stability', jaccard_summary)
show_table('Stability conclusions and leakage checks', stability_contract)

**Selection stability conditions** — 9 row(s)

,condition_id,seed,learners,stage,selection_events,validation_events,KC_count,addition_count,selected_additions,exact_reference_match,validation_log_loss,objective
0,seed_20260827__learners_0060,20260827,60,reference_seed_nested_support,9600,1920,10,1,"[""kc_interaction__aspect_perfect__and__polarity_negative""]",True,0.651563,0.656563
1,seed_20260827__learners_0120,20260827,120,reference_seed_nested_support,19200,3840,10,1,"[""kc_interaction__tense_present__and__voice_passive""]",False,0.652277,0.657277
2,seed_20260827__learners_0240,20260827,240,reference_seed_nested_support,38400,7680,10,1,"[""kc_interaction__aspect_perfect__and__polarity_negative""]",True,0.649995,0.654995
3,seed_20260827__learners_0500,20260827,500,reference_seed_nested_support,80000,16000,10,1,"[""kc_interaction__aspect_perfect__and__polarity_negative""]",True,0.647380,0.652380
4,seed_20260827__learners_1000,20260827,1000,reference_seed_nested_support,160000,32000,10,1,"[""kc_interaction__aspect_perfect__and__polarity_negative""]",True,0.646354,0.651354
5,seed_20260828__learners_1000,20260828,1000,full_support_seed_stability,160000,32000,10,1,"[""kc_interaction__aspect_perfect__and__polarity_negative""]",True,0.646152,0.651152
6,seed_20260829__learners_1000,20260829,1000,full_support_seed_stability,160000,32000,10,1,"[""kc_interaction__aspect_perfect__and__polarity_negative""]",True,0.644646,0.649646
7,seed_20260830__learners_1000,20260830,1000,full_support_seed_stability,160000,32000,10,1,"[""kc_interaction__aspect_perfect__and__polarity_negative""]",True,0.648254,0.653254
8,seed_20260831__learners_1000,20260831,1000,full_support_seed_stability,160000,32000,10,1,"[""kc_interaction__aspect_perfect__and__polarity_negative""]",True,0.646378,0.651378


**Full-support five-seed KC selection frequency** — 10 row(s)

,candidate_id,selected_conditions,selection_frequency,condition_ids
0,kc_feature__aspect__perfect,5,1.0,"[""seed_20260827__learners_1000"", ""seed_20260828__learners_1000"", ""seed_20260829__learners_1000"", ""seed_20260830__lea..."
1,kc_feature__aspect__perfect_progressive,5,1.0,"[""seed_20260827__learners_1000"", ""seed_20260828__learners_1000"", ""seed_20260829__learners_1000"", ""seed_20260830__lea..."
2,kc_feature__aspect__progressive,5,1.0,"[""seed_20260827__learners_1000"", ""seed_20260828__learners_1000"", ""seed_20260829__learners_1000"", ""seed_20260830__lea..."
3,kc_feature__clause__imperative,5,1.0,"[""seed_20260827__learners_1000"", ""seed_20260828__learners_1000"", ""seed_20260829__learners_1000"", ""seed_20260830__lea..."
4,kc_feature__clause__polar_question,5,1.0,"[""seed_20260827__learners_1000"", ""seed_20260828__learners_1000"", ""seed_20260829__learners_1000"", ""seed_20260830__lea..."
5,kc_feature__polarity__negative,5,1.0,"[""seed_20260827__learners_1000"", ""seed_20260828__learners_1000"", ""seed_20260829__learners_1000"", ""seed_20260830__lea..."
6,kc_feature__tense__past,5,1.0,"[""seed_20260827__learners_1000"", ""seed_20260828__learners_1000"", ""seed_20260829__learners_1000"", ""seed_20260830__lea..."
7,kc_feature__tense__present,5,1.0,"[""seed_20260827__learners_1000"", ""seed_20260828__learners_1000"", ""seed_20260829__learners_1000"", ""seed_20260830__lea..."
8,kc_feature__voice__passive,5,1.0,"[""seed_20260827__learners_1000"", ""seed_20260828__learners_1000"", ""seed_20260829__learners_1000"", ""seed_20260830__lea..."
9,kc_interaction__aspect_perfect__and__polarity_negative,5,1.0,"[""seed_20260827__learners_1000"", ""seed_20260828__learners_1000"", ""seed_20260829__learners_1000"", ""seed_20260830__lea..."


**Inventory Jaccard stability** — 6 row(s)

,scope,inventory,mean,median,minimum
0,all_conditions,all_selected,0.959596,1.0,0.818182
1,all_conditions,additions,0.777778,1.0,0.000000
2,reference_nested_support,all_selected,0.927273,1.0,0.818182
3,reference_nested_support,additions,0.600000,1.0,0.000000
4,five_seed_full_support,all_selected,1.000000,1.0,1.000000
5,five_seed_full_support,additions,1.000000,1.0,1.000000


**Stability conclusions and leakage checks** — 1 row(s)

,reference_condition_id,conditions_matching_finalized_reference,full_support_seeds_matching_finalized_reference,all_conditions_identical,all_full_support_seeds_identical,holdout_events_supplied_to_any_selection,reserved_test_events_supplied_to_any_selection
0,seed_20260827__learners_1000,8,5,False,True,0,0


,reference_condition_id,conditions_matching_finalized_reference,full_support_seeds_matching_finalized_reference,all_conditions_identical,all_full_support_seeds_identical,holdout_events_supplied_to_any_selection,reserved_test_events_supplied_to_any_selection
0,seed_20260827__learners_1000,8,5,False,True,0,0


## Final result digest

This final table gathers the principal evidence and its claim boundary. It is a compact reading guide, not a replacement for the stage tables above.

In [17]:
automated_all_logistic = kt_metrics.query(
    "representation == 'automated' and technique == 'logistic' and grammar_regime == 'all_test'"
).iloc[0]
automated_vs_factorized = paired_rows.query(
    "grammar_regime == 'all_test' and reference == 'factorized' and candidate == 'automated'"
).iloc[0]
FINAL_DATASET_SUMMARY = {
    'dataset_id': finalization['dataset_id'],
    'source_descriptors': len(source_rows),
    'canonical_cells': len(canonical_cells),
    'generated_candidates': len(candidate_rows),
    'selected_items': len(selected_bank),
    'learners': events['learner_id'].nunique(),
    'events': len(events),
    'raw_KC_candidates': candidate_inventory['candidate_counts']['raw_total'],
    'selection_eligible_KC_candidates': candidate_inventory['candidate_counts']['selection_eligible'],
    'automatically_selected_KCs': len(selection_trace['selected_candidate_ids']),
}
final_digest = pd.DataFrame([
    {'stage': 'source → canonical', 'result': f"{len(source_rows)} descriptors; {int(mapping_rows['result'].eq('complete').sum())} complete mappings; {len(canonical_cells)} cells", 'claim boundary': 'retained English EGP sample and retained model normalisation'},
    {'stage': 'generation → fixed bank', 'result': f"{len(generation_attempts)} attempts; {len(candidate_rows)} payloads; {len(selected_bank)} selected items; {fold_assignments['cell_id'].nunique()}/{len(canonical_cells)} cells covered", 'claim boundary': 'model-generated and model-judged; packaging correction separately audited'},
    {'stage': 'grammar fold', 'result': '; '.join(f"{row.grammar_split}: {row.cells} cells / {row.selected_items} items" for row in fold_summary.itertuples()), 'claim boundary': 'semantic structural split, not outcome-optimized'},
    {'stage': 'KC candidates', 'result': f"{candidate_inventory['candidate_counts']['raw_total']} raw → {candidate_inventory['candidate_counts']['activation_equivalence_classes']} activation classes → {candidate_inventory['candidate_counts']['selection_eligible']} selection eligible", 'claim boundary': 'equivalence only on the development measurement bank'},
    {'stage': 'automated selection', 'result': f"{len(selection_trace['selected_candidate_ids'])} KCs: nine feature marginals + perfect×negative", 'claim boundary': 'selected on synthetic development learner evidence only'},
    {'stage': 'KT prediction', 'result': f"automated logistic test log loss {automated_all_logistic.log_loss:.6f}; Brier {automated_all_logistic.brier_score:.6f}", 'claim boundary': 'synthetic mixed latent world; observable logistic KT'},
    {'stage': 'paired KC comparison', 'result': f"automated−factorized Δ log loss {automated_vs_factorized.delta_log_loss:.6f} [{automated_vs_factorized.log_loss_CI_low:.6f}, {automated_vs_factorized.log_loss_CI_high:.6f}]", 'claim boundary': 'learner-cluster paired interval; negative favors automated'},
    {'stage': 'selection stability', 'result': f"{stability['exact_inventory_matches']['full_support_seeds_matching_finalized_reference']}/5 full-support seeds match exactly", 'claim boundary': 'one low-support prefix selects a different interaction'},
])
show_table('Machine-readable final counts', pd.DataFrame([FINAL_DATASET_SUMMARY]))
show_table('Evidence-backed final result digest', final_digest)

**Machine-readable final counts** — 1 row(s)

,dataset_id,source_descriptors,canonical_cells,generated_candidates,selected_items,learners,events,raw_KC_candidates,selection_eligible_KC_candidates,automatically_selected_KCs
0,grammar_kt_medium_v1,139,24,77,44,1000,204000,55,28,10


**Evidence-backed final result digest** — 8 row(s)

,stage,result,claim boundary
0,source → canonical,139 descriptors; 44 complete mappings; 24 cells,retained English EGP sample and retained model normalisation
1,generation → fixed bank,78 attempts; 77 payloads; 44 selected items; 24/24 cells covered,model-generated and model-judged; packaging correction separately audited
2,grammar fold,compositional_holdout: 5 cells / 10 items; development: 18 cells / 32 items; novel_feature_holdout: 1 cells / 2 items,"semantic structural split, not outcome-optimized"
3,KC candidates,55 raw → 38 activation classes → 28 selection eligible,equivalence only on the development measurement bank
4,automated selection,10 KCs: nine feature marginals + perfect×negative,selected on synthetic development learner evidence only
5,KT prediction,automated logistic test log loss 0.643356; Brier 0.225610,synthetic mixed latent world; observable logistic KT
6,paired KC comparison,"automated−factorized Δ log loss -0.000375 [-0.000631, -0.000109]",learner-cluster paired interval; negative favors automated
7,selection stability,5/5 full-support seeds match exactly,one low-support prefix selects a different interaction


,stage,result,claim boundary
0,source → canonical,139 descriptors; 44 complete mappings; 24 cells,retained English EGP sample and retained model normalisation
1,generation → fixed bank,78 attempts; 77 payloads; 44 selected items; 24/24 cells covered,model-generated and model-judged; packaging correction separately audited
2,grammar fold,compositional_holdout: 5 cells / 10 items; development: 18 cells / 32 items; novel_feature_holdout: 1 cells / 2 items,"semantic structural split, not outcome-optimized"
3,KC candidates,55 raw → 38 activation classes → 28 selection eligible,equivalence only on the development measurement bank
4,automated selection,10 KCs: nine feature marginals + perfect×negative,selected on synthetic development learner evidence only
5,KT prediction,automated logistic test log loss 0.643356; Brier 0.225610,synthetic mixed latent world; observable logistic KT
6,paired KC comparison,"automated−factorized Δ log loss -0.000375 [-0.000631, -0.000109]",learner-cluster paired interval; negative favors automated
7,selection stability,5/5 full-support seeds match exactly,one low-support prefix selects a different interaction
